# Capítulo 14: Redes Neurais

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 10 de James et al. (2023).

> Como resultado dos ajustes dos pesos, unidades internas "escondidas", que não fazem parte da entrada nem da saída, passam a representar características importantes do domínio da tarefa, e as regularidades da tarefa são capturadas pelas interações dessas unidades.
>
> — David E. Rumelhart, Geoffrey E. Hinton e Ronald J. Williams, em tradução livre (1986, p. 533)

Uma rede neural de camada única calcula $K$ variáveis novas, as ativações $A_k$, cada qual obtida aplicando uma função $g$ fixa a uma combinação linear dos preditores, e prevê com uma combinação linear delas. Com a ReLU, que vale zero de um lado de um ponto e cresce do outro, cada unidade é uma reta dobrada, e a rede soma dobras. A seção 14.1 ajusta cinco unidades ReLU aos anos de carreira dos jogadores de `Hitters` e obtém uma curva linear por partes para o log do salário, com dobras em cerca de 5, 7 e 9 anos, que sobe nos primeiros anos e depois deixa de subir; o MSE de treino cai de 0,5602 com a reta para 0,3940 com a rede. Os pontos de dobra saem dos dados, e com a ativação identidade a rede volta a ser uma reta. A seção 14.2 empilha duas camadas escondidas, com 256 e 128 unidades, sobre os 784 pixels das imagens de dígitos do MNIST e termina numa softmax, que dá uma probabilidade a cada dígito; a rede tem 235.146 pesos e erra 127 das 2.000 imagens de teste, contra 232 da regressão logística e 400 da LDA.

Os pesos de uma rede se ajustam por métodos de gradiente, com o gradiente calculado pela retropropagação, a regra da cadeia aplicada da saída para a entrada. A perda de uma rede não tem um fundo só: na seção 14.3, a mesma rede de cinco unidades, ajustada com o `lbfgs` nos mesmos jogadores, termina em seis valores diferentes de MSE de treino com dez sementes. Cada passo usa um lote de observações, e uma época é uma passada por todas: nas 4.800 imagens de ajuste, com lotes de 200, são 24 passos por época. Acompanhada época a época, a entropia cruzada das imagens de validação chega ao menor valor na época 10 e termina a época 30 acima dele, enquanto a das imagens de ajuste cai a cada época. Parar o ajuste pela validação é a parada antecipada; penalizar a soma dos quadrados dos pesos, com o `alpha` do `scikit-learn` no papel do $\lambda$ da ridge, é a outra regularização medida ali; o *dropout*, que desliga unidades por sorteio, é a terceira, e não existe no `MLPClassifier` nem no `MLPRegressor`. Nos dígitos, parar pela perda de validação deu probabilidades melhores no teste e mais erros: o critério de parada tem de medir o que se quer do modelo, a probabilidade ou o acerto.

A seção 14.4 monta a rede para prever o salário dos jogadores em milhares de dólares. Sem padronizar os preditores, a rede erra em média 56.829 mil dólares; sem padronizar a resposta, nem 2.000 épocas bastam para ela passar da regressão linear. Com as duas escalas no `Pipeline`, a validação cruzada escolhe o número de unidades e o `alpha` juntos, e a rede escolhida erra, em média, 178,34 mil dólares nos jogadores de teste, contra 232,06 da regressão linear; nos pacientes de `Heart`, a rede erra um paciente de teste a mais que a regressão logística. A seção 14.5 pergunta quando o trabalho compensa. Em 20 divisões dos jogadores, a rede erra menos que os modelos lineares em todas, e uma floresta aleatória, sem busca nem padronização, erra menos que a rede em todas; nas imagens, a rede, na média de cinco sementes, erra menos que a floresta e que a logística em todos os tamanhos de treino medidos, de 150 a 6.000 imagens. As redes deste capítulo são densas: cada unidade recebe todas as entradas, e nada nelas usa a vizinhança dos pixels ou a ordem de uma sequência. Elas se ajustam com o `scikit-learn`. As redes convolucionais, para imagens, e as recorrentes, para textos e séries no tempo, se ajustam com bibliotecas de tensores, em geral com placas de vídeo.

Ao final deste capítulo, você será capaz de:

- Escrever a previsão de uma rede de camada única a partir dos pesos, reconstruí-la com `coefs_` e `intercepts_` de um `MLPRegressor` e explicar por que a ativação não pode ser linear
- Descrever uma rede multicamada com saída softmax, contar os pesos dela e ajustar um `MLPClassifier` a imagens, comparando-o com a LDA e a regressão logística
- Explicar o gradiente descendente, a retropropagação, os lotes e as épocas, mostrar que a perda tem mais de um mínimo e acompanhar época a época, com `partial_fit`, as perdas de ajuste e de validação
- Regularizar uma rede com a penalidade `alpha` e com a parada antecipada, e converter o `alpha` do `scikit-learn` no $\lambda$ da fórmula
- Padronizar os preditores e a resposta, ler um `ConvergenceWarning`, escolher o tamanho da camada e a penalidade por validação cruzada e decidir, contra modelos mais simples, se a rede compensa e por que uma rede densa não usa a vizinhança dos pixels

## Seções

| Seção | Tópico |
|---|---|
| [14.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/01-uma-rede-de-camada-unica.html) | Uma Rede de Camada Única |
| [14.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/02-redes-multicamada.html) | Redes Multicamada |
| [14.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/03-ajustando-uma-rede.html) | Ajustando uma Rede: Retropropagação, Gradiente Estocástico e Regularização |
| [14.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-mlpclassifier-e-mlpregressor-na-pratica.html) | MLPClassifier e MLPRegressor na Prática |
| [14.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/05-quando-usar-deep-learning.html) | Quando Usar Deep Learning |

## Uma Rede de Camada Única

> **📌 Nota**
>
> Esta seção corresponde à seção 10.1 de James et al. (2023).

O salário de um jogador de beisebol cresce com os anos de carreira, mas não em linha reta: depois de alguns anos ele para de subir. Uma reta ajustada a esses dados sobe com a mesma inclinação do começo ao fim. Dá para montar uma curva que acompanhe o dado somando algumas retas "dobradas", cada uma ligada só de um lado de certo ponto?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Os anos de carreira

A tabela `Hitters` reúne estatísticas de jogadores das grandes ligas de beisebol dos Estados Unidos na temporada de 1986, com o salário da temporada de 1987 em milhares de dólares. O único preditor aqui é `anos`, o número de anos do jogador na liga principal. A resposta é o logaritmo natural do salário, e o motivo é a escala: os salários variam muito de um jogador para outro, e no log uma mesma diferença corresponde a uma mesma proporção do salário, seja ele pequeno ou grande.

O código lê a tabela, descarta os jogadores sem salário e calcula a média do log do salário em cinco faixas de anos de carreira. Ao lado de cada média vai $e$ elevado a ela, que dá um salário típico da faixa, em milhares de dólares:

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv")
print("jogadores na tabela:", len(hitters))
hitters = hitters.dropna()
print("com salário:", len(hitters))
print("anos: de", hitters["anos"].min(), "a", hitters["anos"].max())
print("salário: de", numero_br(hitters["salario"].min(), 1),
      "a", numero_br(hitters["salario"].max(), 1))
y = np.log(hitters["salario"])
X = hitters[["anos"]]
limites = [0, 3, 6, 9, 12, 24]
faixas = pd.cut(hitters["anos"], limites,
                labels=["1 a 3", "4 a 6", "7 a 9",
                        "10 a 12", "13 a 24"])
medias = y.groupby(faixas, observed=True).mean()
print("média do log (e^média):")
for faixa, media in medias.items():
    print(f"  {faixa} anos: {numero_br(media)}"
          f" ({numero_br(np.exp(media), 0)})")

> **🔧 Função**
>
> **`pd.read_csv(caminho)`** — lê um arquivo CSV numa tabela (`DataFrame`). **`df.dropna()`** — a tabela sem as linhas que têm algum valor ausente. **`serie.min()`**, **`serie.max()`** — o menor e o maior valor da coluna. **`np.log(v)`** — o logaritmo natural de cada valor; **`np.exp(v)`** faz o caminho de volta, $e$ elevado a cada valor. **`pd.cut(v, limites, labels)`** — troca cada valor de `v` pela faixa em que ele cai; cada faixa inclui o limite da direita e exclui o da esquerda, e `labels` dá o nome das faixas. **`serie.groupby(faixas, observed=True).mean()`** — a média de `serie` em cada faixa; `observed=True` deixa de fora faixas sem nenhum jogador. **`serie.items()`** — percorre a série em pares (rótulo, valor).

Dos 322 jogadores, 263 têm salário, com carreiras de 1 a 24 anos e salários de 67,5 a 2.460,0 mil dólares. A média do log do salário sobe de 4,89 nos três primeiros anos (um salário típico de cerca de 133 mil dólares) para 5,97 entre o quarto e o sexto (390 mil) e para 6,45 entre o sétimo e o nono (636 mil). Nas duas faixas seguintes, ela fica em 6,37 e 6,47. A figura mostra os jogadores e essas médias:

In [ ]:
# Figura: Log do salário contra os anos de carreira dos 263 jogadores (pontos cinza, deslocados um pouco na horizontal para não se sobreporem) e a média do log do salário em cada faixa de anos (segmentos azuis).
rng = np.random.default_rng(14)
tremor = rng.uniform(-0.25, 0.25, size=len(X))
fig, ax = plt.subplots(figsize=(3.8, 3.2))
ax.scatter(X["anos"] + tremor, y, s=10, color="0.45", alpha=0.4,
           linewidths=0)
for inicio, fim, media in zip(limites[:-1], limites[1:], medias):
    ax.plot([inicio + 0.5, fim + 0.5], [media, media], color="C0",
            linewidth=2.5)
ax.set_xlim(0, 25)
ax.set_xlabel("anos de carreira")
ax.set_ylabel("log do salário")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:g}".replace(".", ","))
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.random.default_rng(14)`** — um gerador de números aleatórios com semente 14; **`rng.uniform(a, b, size)`** sorteia `size` números uniformes entre `a` e `b`, aqui só para afastar na horizontal os pontos de jogadores com os mesmos anos. **`plt.subplots(figsize=(largura, altura))`** — cria uma figura, com o tamanho em polegadas, e devolve a figura e o eixo `ax` em que se desenha. **`ax.scatter(x, y, s, color, alpha, linewidths)`** — um ponto por par `(x, y)`, de área `s`, com transparência `alpha` e sem contorno (`linewidths=0`). **`ax.plot(x, y, color, linewidth)`** — uma linha pelos pontos dados, aqui um segmento de dois pontos. **`ax.set_xlim(a, b)`** — os limites do eixo horizontal; `set_ylim` faz o mesmo no vertical. **`ax.set_xlabel(texto)`**, **`ax.set_ylabel(texto)`** — os nomes dos eixos. **`ax.yaxis.set_major_formatter(f)`** — escreve cada marca do eixo vertical com a função `f`, aqui trocando o ponto decimal pela vírgula. **`plt.tight_layout()`** — ajusta as margens para nada ficar cortado. **`plt.show()`** — mostra a figura.

O crescimento dos primeiros anos não continua, e é essa forma que uma reta não descreve.

### A rede como soma de transformações

Uma rede neural monta uma função não linear $f(X)$ dos preditores $X = (X_1, \dots, X_p)$ a partir de $K$ variáveis novas. Cada uma sai de uma combinação linear dos preditores, passada por uma função $g$ fixada de antemão, e $f(X)$ é uma combinação linear dessas $K$ variáveis. O diagrama mostra o caminho, com $p = 4$ preditores e $K = 5$ variáveis novas:

In [ ]:
# Figura: Uma rede de camada única com $p = 4$ preditores e $K = 5$ unidades escondidas. Cada preditor $X_j$ (cinza) alimenta cada unidade $A_k$ (azul) com um peso $w_{kj}$, e cada unidade alimenta a saída $f(X)$ (laranja) com um peso $\beta_k$.
camadas = [
    (0, [1.5, 0.5, -0.5, -1.5], "X", "0.45"),
    (1, [2, 1, 0, -1, -2], "A", "C0"),
    (2, [0], None, "C1"),
]
seta = dict(arrowstyle="-|>", color="0.45", linewidth=0.6,
            shrinkA=10, shrinkB=10)
fig, ax = plt.subplots(figsize=(3.8, 3.0))
for (x0, ys0, _, _), (x1, ys1, _, _) in zip(camadas[:-1], camadas[1:]):
    for y0 in ys0:
        for y1 in ys1:
            ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=seta)
for x, ys, letra, cor in camadas:
    ax.scatter([x] * len(ys), ys, s=420, facecolors="none",
               edgecolors=cor, linewidths=1.5)
    if letra:
        for k, y_no in enumerate(ys, start=1):
            ax.text(x, y_no, f"${letra}_{k}$", ha="center", va="center")
ax.text(2.2, 0, "$f(X)$", ha="left", va="center")
for x, nome in [(0, "entrada"), (1, "escondida"), (2, "saída")]:
    ax.text(x, 2.85, nome, ha="center")
ax.text(0.5, -2.7, "$w_{kj}$", ha="center")
ax.text(1.5, -2.7, "$\\beta_k$", ha="center")
ax.set_xlim(-0.35, 2.75)
ax.set_ylim(-2.9, 3.2)
ax.axis("off")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.annotate("", xy, xytext, arrowprops)`** — desenha uma seta de `xytext` até `xy`, sem texto; `arrowprops` dá o formato da seta, e `shrinkA`/`shrinkB` a encurtam nas pontas, em pontos, para não entrar nos círculos. **`ax.scatter(..., facecolors="none", edgecolors=cor)`** — pontos só com o contorno, aqui os círculos do diagrama. **`ax.text(x, y, texto, ha, va)`** — escreve `texto` na posição `(x, y)`; `ha` e `va` dizem como alinhá-lo na horizontal e na vertical, e o que vai entre `$` sai como fórmula. **`ax.axis("off")`** — esconde os eixos.

A coluna da esquerda é a ***camada de entrada***, com os preditores. A do meio é a ***camada escondida***, e cada círculo dela é uma ***unidade escondida*** (*hidden unit*): as unidades não são observadas nos dados, são calculadas pela rede. A da direita é a ***camada de saída***, com a previsão. Cada seta leva um peso.

> **🔷 Conceito**
>
> Uma ***rede neural de camada única*** com $K$ unidades escondidas prevê a resposta por
> $$
> f(X) = \beta_0 + \sum_{k=1}^{K} \beta_k A_k,
> $$
> em que cada unidade calcula
> $$
> A_k = g\Big(w_{k0} + \sum_{j=1}^{p} w_{kj} X_j\Big).
> $$
>
> Os $A_k$ são as ***ativações*** das unidades, e $g$ é a ***função de ativação***. Os números $w_{k0}, \dots, w_{kp}$ e $\beta_0, \dots, \beta_K$ são os ***pesos*** da rede; $w_{k0}$ e $\beta_0$ são os interceptos. Todos são estimados juntos, a partir dos dados.

Uma unidade de cada vez. Com um preditor só, os anos de carreira, tome a unidade de pesos $w_{10} = -5$ e $w_{11} = 1$, e como $g$ a função que devolve zero para um número negativo e o próprio número para um positivo. A unidade calcula $A_1 = g(-5 + \text{anos})$. A conta para os três primeiros jogadores da tabela:

In [ ]:
ativacao = lambda z: np.maximum(0, z)
w_0, w_1 = -5, 1
for jogador, anos in hitters["anos"].iloc[:3].items():
    print(f"jogador {jogador}: {anos} anos,",
          f"A = {ativacao(w_0 + w_1 * anos)}")

> **🔧 Função**
>
> **`serie.iloc[:3]`** — as três primeiras posições da série, qualquer que seja o rótulo delas. **`np.maximum(0, v)`** — o maior entre 0 e cada valor de `v`.

O jogador de 3 anos tem $A_1 = 0$: para ele, a unidade está desligada. Os de 14 e de 11 anos têm $A_1 = 9$ e $A_1 = 6$, os anos que passam de 5. A unidade mede quanto a carreira passa de 5 anos, e o ponto 5 vem dos pesos $w_{10}$ e $w_{11}$. Outra unidade, com outros pesos, liga noutro ponto, e a saída soma as unidades, cada uma multiplicada pelo seu $\beta_k$.

Com os $w$ fixados, $f(X)$ é uma regressão linear nas ativações; a diferença para uma regressão em variáveis transformadas, como $X$ e $X^2$, é que quem ajusta não escolhe as transformações: os $w$, e com eles os pontos de dobra, também são estimados a partir dos dados. O nome "neural" vem de uma analogia com o cérebro: cada unidade faria o papel de um neurônio, que "dispara" quando a ativação é grande.

#### Duas funções de ativação

A ***sigmoide***,
$$
g(z) = \frac{e^z}{1 + e^z} = \frac{1}{1 + e^{-z}},
$$
é a mesma função que a regressão logística do capítulo 9 usa para levar uma combinação linear a uma probabilidade entre 0 e 1. A ***ReLU*** (*rectified linear unit*) é a função da unidade acima: vale zero para $z$ negativo e o próprio $z$ para $z$ positivo,
$$
g(z) = (z)_+ = \begin{cases} 0 & \text{se } z < 0,\\ z & \text{se } z \ge 0. \end{cases}
$$
É muito usada nas redes atuais, e é a que o `scikit-learn` usa se nada for dito. As duas lado a lado:

In [ ]:
# Figura: As duas funções de ativação, em $z$ de −4 a 4: a sigmoide (em cima, azul), que vai de 0 a 1, e a ReLU (embaixo, laranja), que vale zero para $z$ negativo e $z$ para $z$ positivo.
z = np.linspace(-4, 4, 401)
sigmoide = 1 / (1 + np.exp(-z))
relu = np.maximum(0, z)
fig, (ax_s, ax_r) = plt.subplots(2, 1, figsize=(3.8, 4.4),
                                 sharex=True)
ax_s.plot(z, sigmoide, color="C0")
ax_s.set_title("sigmoide")
ax_s.set_ylabel("g(z)")
ax_r.plot(z, relu, color="C1")
ax_r.set_title("ReLU")
ax_r.set_ylabel("g(z)")
ax_r.set_xlabel("z")
ax_r.set_xlim(-4.2, 4.2)
ax_s.set_ylim(-0.05, 1.05)
ax_r.set_ylim(-0.2, 4.2)
for ax in (ax_s, ax_r):
    ax.yaxis.set_major_formatter(
        lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))
ax_r.xaxis.set_major_formatter(
    lambda v, _: f"{v:g}".replace("-", "−"))
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.linspace(inicio, fim, n)`** — `n` números igualmente espaçados de `inicio` a `fim`, inclusive. **`plt.subplots(2, 1, sharex=True)`** — uma figura com dois gráficos empilhados que dividem o mesmo eixo horizontal. **`ax.set_title(texto)`** — o título de um gráfico.

A ReLU é uma reta "dobrada" em $z = 0$: desligada de um lado, ligada do outro. Numa unidade, a dobra fica onde $w_{k0} + \sum_j w_{kj} X_j = 0$, e o peso $w_{k0}$ a desloca para onde for preciso; na unidade do exemplo, ela fica em 5 anos.

### Uma rede nos anos de carreira

Com um preditor só, cada unidade de uma rede ReLU é uma reta dobrada no eixo dos anos. A rede abaixo tem $K = 5$ unidades ReLU, é ajustada ao log do salário com `anos` padronizado e é comparada com a reta de mínimos quadrados pelo MSE de treino, calculado nos mesmos 263 jogadores do ajuste.

O motivo da padronização está no começo da busca pelos pesos: os pesos iniciais são sorteados numa faixa que depende só do número de entradas e de unidades, não da escala de `anos`. A dobra de uma unidade fica em $-w_{k0}/w_{k1}$, e com os dois pesos sorteados na mesma faixa as dobras iniciais tendem a cair perto de zero anos, na beira dos dados. Com `anos` padronizado, elas tendem a cair perto da média, no meio dos jogadores:

In [ ]:
rede = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(5,), solver="lbfgs",
                 max_iter=5000, random_state=14))
rede.fit(X, y)
reta = LinearRegression().fit(X, y)
mse_reta = mean_squared_error(y, reta.predict(X))
mse_rede = mean_squared_error(y, rede.predict(X))
print("MSE de treino, reta:", numero_br(mse_reta, 4))
print("MSE de treino, rede:", numero_br(mse_rede, 4))
inclinacao = reta.coef_[0]
print("inclinação da reta:", numero_br(inclinacao))
print("e^inclinação:", numero_br(np.exp(inclinacao), 3))

> **🔧 Função**
>
> **`MLPRegressor(...)`** — a rede neural de regressão do `scikit-learn`. Os parâmetros usados aqui:
>
> - `hidden_layer_sizes`: o número de unidades de cada camada escondida, numa tupla; `(5,)` é uma camada com $K = 5$. **Sempre por nome**: o primeiro parâmetro posicional do `MLPRegressor` é outro (`loss`), e `MLPRegressor((5,))` levanta erro ao ajustar.
> - `solver`: o método que procura os pesos. `"lbfgs"` usa todos os dados a cada passo; o ajuste por lotes de dados, que é o do padrão, é assunto da seção 14.3.
> - `max_iter`: o número máximo de passos do método.
> - `random_state`: fixa o sorteio dos pesos iniciais.
>
> **`StandardScaler()`** — padroniza cada coluna: subtrai a média e divide pelo desvio padrão, com a média e o desvio calculados nos dados do `fit`. **`make_pipeline(passo1, passo2)`** — encadeia os passos: o `fit` ajusta o primeiro, transforma os dados e ajusta o segundo no resultado, e o `predict` transforma antes de prever. **`modelo.fit(X, y)`** — estima os pesos do modelo e devolve o próprio modelo. **`modelo.predict(X)`** — a previsão para cada linha de `X`. **`LinearRegression()`** — a regressão linear por mínimos quadrados; **`reta.coef_`** guarda a inclinação, uma por preditor. **`mean_squared_error(y, previsto)`** — o MSE, a média de $(y_i - \hat y_i)^2$.

O MSE de treino cai de 0,5602 com a reta para 0,3940 com a rede. As duas medidas são de treino, nos jogadores usados no ajuste. A rede consegue reproduzir qualquer reta (basta uma unidade ligada para todos os jogadores) e tem mais pesos, então ela tende a ficar mais perto desses jogadores; se ela prevê melhor jogadores novos é outra pergunta, que pede dados de fora do ajuste.

Quantos pesos a rede tem, e com que penalidade ela foi ajustada:

In [ ]:
mlp = rede[-1]
pesos = (sum(W.size for W in mlp.coefs_)
         + sum(b.size for b in mlp.intercepts_))
print("pesos da rede:", pesos)
print("alpha:", numero_br(mlp.alpha, 4))

> **🔧 Função**
>
> **`pipeline[-1]`** — o último passo de um pipeline, aqui a rede já ajustada; `pipeline[0]` é o primeiro. **`mlp.coefs_`** — os pesos $w_{kj}$ e $\beta_k$ sem os interceptos, numa lista de tabelas: `coefs_[0]` tem uma linha por preditor e uma coluna por unidade escondida, e `coefs_[1]`, uma linha por unidade e uma coluna por saída. **`mlp.intercepts_`** — os interceptos: `intercepts_[0]` tem os $w_{k0}$, e `intercepts_[1]`, o $\beta_0$. **`tabela.size`** — o número de valores de uma tabela do `numpy`. **`mlp.alpha`** — a força da penalidade sobre os pesos, que aparece no fim da seção; o padrão é o valor impresso.

A rede tem 16 pesos: cada uma das $K = 5$ unidades tem um intercepto e um peso para o único preditor, $p = 1$, e a saída tem um intercepto e um peso por unidade, isto é, $K(p + 1) + K + 1 = 5 \times 2 + 5 + 1 = 16$. A reta tem dois: o intercepto e a inclinação.

A rede ajustada é a fórmula da definição, com os pesos de `coefs_` e `intercepts_`. O código abaixo refaz a previsão à mão, com `np.maximum` no lugar de $g$, e confere que ela é igual à do `predict`. Depois calcula, para cada unidade, o ponto de dobra em anos de carreira, o lado em que ela liga, o sinal de $\beta_k$ e se ela está ligada para algum dos 263 jogadores:

In [ ]:
escala = rede[0]
W1, W2 = mlp.coefs_
b1, b2 = mlp.intercepts_
x_til = escala.transform(X)
A = np.maximum(0, x_til @ W1 + b1)
f_mao = (A @ W2 + b2).ravel()
print("à mão = predict:", np.allclose(f_mao, rede.predict(X)))
dobra_til = -b1 / W1[0]
coluna = dobra_til.reshape(-1, 1)
dobra = escala.inverse_transform(coluna)
dobra = dobra.ravel()
unidades = pd.DataFrame({
    "dobra": [numero_br(d, 0) for d in dobra],
    "liga": np.where(W1[0] > 0, "acima", "abaixo"),
    "β": np.where(W2[:, 0] > 0, "+", "−"),
    "ativa": (A > 0).any(axis=0),
}, index=pd.RangeIndex(1, 6, name="unidade"))
print(unidades.to_string())
print("unidades ativas:", unidades["ativa"].sum())

> **🔧 Função**
>
> **`escala.transform(X)`** — aplica a padronização aprendida no `fit`. **`escala.inverse_transform(v)`** — desfaz a padronização, levando valores da escala da rede de volta para anos; pede uma tabela de uma coluna, e `reshape(-1, 1)` põe `v` nesse formato. **`a @ b`** — o produto de matrizes. **`v.ravel()`** — a tabela como um vetor de uma dimensão. **`np.allclose(a, b)`** — `True` se `a` e `b` são iguais posição a posição, a menos de uma diferença muito pequena. **`np.where(condicao, a, b)`** — `a` onde a condição vale `True`, `b` onde vale `False`. **`(A > 0).any(axis=0)`** — para cada coluna, `True` se algum valor dela é `True`. **`pd.DataFrame({nome: valores}, index)`** — monta uma tabela com uma coluna por entrada do dicionário. **`pd.RangeIndex(1, 6, name="unidade")`** — o índice de 1 a 5, com nome. **`df.to_string()`** — a tabela inteira como texto. **`serie.sum()`** — a soma dos valores; numa série de `True` e `False`, conta os `True`.

A soma refeita à mão é a previsão da rede. Seja $\tilde x$ o número de anos padronizado, que é o que a rede recebe. A dobra de cada unidade está onde $w_{k0} + w_{k1} \tilde x = 0$, isto é, em $\tilde x = -w_{k0}/w_{k1}$, levado de volta para anos. Se $w_{k1}$ é positivo, a unidade está desligada abaixo da dobra e ligada acima dela; se é negativo, o contrário.

Os pesos dependem do sorteio inicial, fixado por `random_state`: com outra semente, a numeração das unidades e a posição das dobras podem mudar, e o que se mantém é a forma linear por partes. Nesta semente, a tabela mostra quatro unidades ativas. A unidade 1 dobra em −20 anos e liga acima: para todos os jogadores ela está no trecho ligado, e é uma reta que sobe, com $\beta_1$ positivo. As unidades 2 e 3 ligam acima de 7 e de 9 anos, com $\beta$ negativo: a partir dali, tiram inclinação da curva. A unidade 4 liga abaixo de 5 anos, também com $\beta$ negativo; a figura mais adiante mostra o que ela faz. A unidade 5 dobra perto de 0 e liga abaixo, e nenhum jogador tem menos de um ano de carreira: ela nunca liga, e o peso $\beta_5$ não entra em nenhuma previsão.

A curva da rede e a reta, sobre os jogadores:

In [ ]:
# Figura: Log do salário contra os anos de carreira dos 263 jogadores (pontos cinza), com a previsão da rede de cinco unidades ReLU (laranja) e a da reta de mínimos quadrados (cinza tracejado).
grade = pd.DataFrame({"anos": np.linspace(1, 24, 300)})
f_rede = rede.predict(grade)
f_reta = reta.predict(grade)
anos_inteiros = pd.DataFrame({"anos": np.arange(1, 25)})
f_inteiros = rede.predict(anos_inteiros)
maior = np.argmax(f_inteiros)
print("anos do maior f̂:", maior + 1)
for i in (0, maior, 23):
    anos = "1 ano" if i == 0 else f"{i + 1} anos"
    print(f"f̂ em {anos}: {numero_br(f_inteiros[i])}"
          f" ({numero_br(np.exp(f_inteiros[i]), 0)})")
fig, ax = plt.subplots(figsize=(3.8, 3.2))
ax.scatter(X["anos"] + tremor, y, s=10, color="0.45", alpha=0.4,
           linewidths=0, label="jogadores")
ax.plot(grade["anos"], f_rede, color="C1", linewidth=2.5,
        label="rede")
ax.plot(grade["anos"], f_reta, color="0.45", linestyle="--",
        label="reta")
ax.set_xlim(0, 25)
ax.set_xlabel("anos de carreira")
ax.set_ylabel("log do salário")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:g}".replace(".", ","))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.arange(1, 25)`** — os inteiros de 1 a 24. **`np.argmax(v)`** — a posição do maior valor de `v`, contada a partir de 0. **`ax.plot(..., linestyle="--", label)`** — `linestyle` escolhe o traço, e `label` é o nome da linha na legenda. **`ax.legend(loc, bbox_to_anchor, ncol)`** — a legenda com os `label` das linhas e pontos; aqui ela é presa pela borda de baixo (`loc`) no topo do gráfico (`bbox_to_anchor`), em `ncol` colunas.

A rede prevê 4,53 para um jogador de 1 ano, um salário típico de cerca de 93 mil dólares, sobe até 6,51 em 9 anos (672 mil) e termina em 6,26 em 24 anos (521 mil). A reta não tem como fazer esse movimento: sobe 0,10 no log do salário a cada ano de carreira, do primeiro ao último. Como $e$ elevado à inclinação vale 1,105, isso é um salário 10,5% maior a cada ano.

Onde está cada dobra na curva? A curva é $\hat f(x) = \beta_0 + \sum_k \beta_k A_k(x)$, e cada parcela $\beta_k A_k(x)$ é uma reta dobrada: zero de um lado da dobra e uma reta do outro. As parcelas das quatro unidades ativas e a soma delas, $\hat f(x) - \beta_0$, ficam assim:

In [ ]:
# Figura: As parcelas $\beta_k A_k(x)$ das quatro unidades ativas da rede (azul, finas, numeradas como na tabela) contra os anos de carreira, e a soma delas, $\hat f(x) - \beta_0$ (laranja, grossa). Cada parcela é zero de um lado da dobra da unidade e uma reta do outro; as linhas pontilhadas marcam as dobras que caem entre 1 e 24 anos, com o número da unidade no alto.
anos_grade = grade[["anos"]]
A_grade = np.maximum(0, escala.transform(anos_grade) @ W1 + b1)
parcelas = A_grade * W2[:, 0]
ativas = np.flatnonzero(unidades["ativa"])
fig, ax = plt.subplots(figsize=(3.8, 3.4))
ax.axhline(0, color="0.45", linewidth=0.5, alpha=0.6, zorder=0)
for k in ativas:
    if 1 < dobra[k] < 24:
        ax.axvline(dobra[k], ymax=0.9, color="0.45", linewidth=0.8,
                   linestyle=":")
        ax.text(dobra[k], 0.93, str(k + 1), ha="center",
                transform=ax.get_xaxis_transform())
    ax.plot(grade["anos"], parcelas[:, k], color="C0", linewidth=1,
            label="parcelas" if k == ativas[0] else "_nolegend_")
    fim = -1 if W1[0, k] > 0 else 0
    ax.annotate(str(k + 1), (grade["anos"].iloc[fim], parcelas[fim, k]),
                xytext=(4 if fim == -1 else -8, 0),
                textcoords="offset points", va="center", color="C0")
ax.plot(grade["anos"], parcelas.sum(axis=1), color="C1", linewidth=2.5,
        label="soma")
ax.set_xlim(-1.5, 26.5)
ax.set_xlabel("anos de carreira")
ax.set_ylabel("parcela no log do salário")
ax.yaxis.set_major_formatter(
    lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.flatnonzero(m)`** — as posições em que `m` vale `True`, contadas a partir de 0. **`ax.axhline(y)`** — uma linha horizontal na altura `y`; `zorder=0` a põe atrás das outras. **`ax.axvline(x, ymax)`** — uma linha vertical na posição `x`, que sobe até a fração `ymax` da altura do gráfico. **`transform=ax.get_xaxis_transform()`** — no `ax.text`, lê `x` em anos e `y` como fração da altura do gráfico. **`ax.annotate(texto, (x, y), ...)`** — com texto, escreve `texto` perto do ponto `(x, y)`; `textcoords="offset points"` lê `xytext` como deslocamento em pontos de tipografia. Em `iloc`, a posição `-1` é a última.

A parcela 1 é a reta que sobe em todo o eixo. A parcela 4 é negativa nos primeiros anos e sobe até zero na dobra dela: rebaixa o nível da curva nos primeiros anos e, ao mesmo tempo, soma inclinação, de modo que, até ali, a soma sobe mais depressa que a parcela 1 sozinha. A partir das dobras das unidades 2 e 3, as duas parcelas negativas vão tirando inclinação da soma. Entre duas dobras vizinhas, todas as parcelas são retas, e a soma também; em cada dobra, a inclinação da soma muda.

### Por que a ativação não pode ser linear

Com $g(z) = z$, a função identidade, a rede de um preditor seria
$$
\begin{aligned}
f(X) &= \beta_0 + \sum_{k=1}^{K} \beta_k (w_{k0} + w_{k1} X_1)\\
&= \Big(\beta_0 + \sum_{k=1}^{K} \beta_k w_{k0}\Big)\\
&\quad + \Big(\sum_{k=1}^{K} \beta_k w_{k1}\Big) X_1,
\end{aligned}
$$
um intercepto mais uma inclinação vezes $X_1$: uma reta, qualquer que seja $K$. O código ajusta aos anos de carreira uma rede de cinco unidades com a ativação identidade e compara as previsões dela com as da reta:

In [ ]:
identidade = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(5,), activation="identity",
                 solver="lbfgs", max_iter=5000, tol=1e-10,
                 random_state=14))
identidade.fit(X, y)
iguais = np.allclose(identidade.predict(X), reta.predict(X),
                     atol=1e-4)
print("rede identidade = reta:", iguais)

> **🔧 Função**
>
> Dois parâmetros novos do **`MLPRegressor`**: `activation` é a função $g$, com `"relu"` como padrão e `"identity"` para $g(z) = z$; `tol` é a tolerância da parada. Com `"lbfgs"`, o método para quando o gradiente da perda fica abaixo de `tol`, ou antes, por critérios próprios do método, como a perda deixar de cair de um passo para o outro. **`np.allclose(a, b, atol)`** — com `atol`, `True` se `a` e `b` diferem, posição a posição, em no máximo `atol` (mais uma pequena parcela relativa).

A rede de cinco unidades com a ativação identidade prevê, nos 263 jogadores, o mesmo que a reta, a menos de 0,0002.

A não linearidade também cria interações. Tome $p = 2$ preditores, $K = 2$ unidades e $g(z) = z^2$, com os pesos
$$
\begin{aligned}
&\beta_0 = 0,\ \beta_1 = \tfrac14,\ \beta_2 = -\tfrac14;\\
&w_{10} = 0,\ w_{11} = 1,\ w_{12} = 1;\\
&w_{20} = 0,\ w_{21} = 1,\ w_{22} = -1.
\end{aligned}
$$
A primeira unidade calcula $A_1 = (X_1 + X_2)^2$, e a segunda, $A_2 = (X_1 - X_2)^2$. Nenhuma delas multiplica $X_1$ por $X_2$. Antes de rodar o código, um palpite: o que sai de $f(X) = \tfrac14 A_1 - \tfrac14 A_2$? O cálculo abaixo avalia $f$ numa grade de 5 × 5 valores de $X_1$ e $X_2$ entre −2 e 2 e compara com o produto $X_1 X_2$:

In [ ]:
x1, x2 = np.meshgrid(np.linspace(-2, 2, 5), np.linspace(-2, 2, 5))
g = lambda z: z ** 2
A1 = g(0 + 1 * x1 + 1 * x2)
A2 = g(0 + 1 * x1 - 1 * x2)
f = 0 + 0.25 * A1 - 0.25 * A2
print("pontos da grade:", f.size)
print("f = X1 · X2 em todos:", np.allclose(f, x1 * x2))

> **🔧 Função**
>
> **`np.meshgrid(a, b)`** — duas tabelas com todas as combinações de um valor de `a` com um valor de `b`: a primeira repete `a` em cada linha, a segunda repete `b` em cada coluna.

Nos 25 pontos, $f$ é o produto $X_1 X_2$. A conta explica: abrindo os quadrados,
$$
\begin{aligned}
f(X) &= \tfrac14 (X_1 + X_2)^2\\
&\quad - \tfrac14 (X_1 - X_2)^2\\
&= \tfrac14 \cdot 4 X_1 X_2\\
&= X_1 X_2,
\end{aligned}
$$
porque os termos $X_1^2$ e $X_2^2$ se cancelam. O quadrado serve para o exemplo, mas não para uma rede de verdade: com $g(z) = z^2$, a rede é sempre um polinômio de grau no máximo 2 nos preditores. A sigmoide e a ReLU não têm esse limite.

### A curva que a rede montou

A pergunta do começo tem resposta: a rede de cinco unidades ReLU montou, para o log do salário, uma curva linear por partes, com dobras em cerca de 5, 7 e 9 anos de carreira, e as dobras e as inclinações saíram dos dados. O número de unidades, $K = 5$, não saiu: foi escolhido. Com mais unidades cabem mais dobras, e o MSE de treino não tem como mostrar se elas descrevem o salário ou só os 263 jogadores do ajuste; $K$ se escolhe e se julga pelo erro em dados de fora do ajuste.

Os pesos foram estimados para que a rede acerte os jogadores do treino: o ajuste procura os $w$ e os $\beta$ que tornam pequena a soma dos quadrados dos resíduos (RSS),
$$
\mathrm{RSS} = \sum_{i=1}^{n} \big(y_i - \hat f(x_i)\big)^2,
$$
que é $n$ vezes o MSE de treino. O `MLPRegressor` acrescenta ao RSS uma penalidade de força `alpha`, 0,0001 no padrão, sobre os pesos $w_{kj}$ e $\beta_k$; os interceptos $w_{k0}$ e $\beta_0$ ficam de fora dela. O assunto da seção 14.3 é como essa busca é feita e por que ela pode terminar em pesos diferentes conforme o ponto de partida.

## Redes Multicamada

> **📌 Nota**
>
> Esta seção corresponde à seção 10.2 de James et al. (2023).

Uma imagem de 28 × 28 pixels de um dígito escrito à mão é feita de 784 números, um por pixel. Um modelo linear nesses 784 números, como a regressão logística, reconhece o dígito com que taxa de erro? E uma rede com duas camadas escondidas, que monta as próprias variáveis a partir dos pixels?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.neural_network import MLPClassifier

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### As imagens

O MNIST é uma coleção de dígitos de 0 a 9 escritos à mão por pessoas diferentes e digitalizados em imagens de 28 × 28 pixels, em tons de cinza. Cada pixel é um inteiro de 0, o fundo, a 255, o traço mais escuro. As duas tabelas usadas aqui trazem as primeiras 6.000 imagens do conjunto de treino do MNIST e as primeiras 2.000 do conjunto de teste, uma imagem por linha: a coluna `digito` tem o rótulo, e as colunas `pixel_000` a `pixel_783` têm os pixels, lidos linha a linha da imagem, da esquerda para a direita.

O código lê as duas tabelas, separa o rótulo dos pixels e divide os pixels por 255, para que fiquem entre 0 e 1:

In [ ]:
treino = pd.read_csv("dados/mnist_treino.csv.gz")
teste = pd.read_csv("dados/mnist_teste.csv.gz")
print("treino:", treino.shape)
print("teste:", teste.shape)
pixels = treino.drop(columns="digito")
print("pixels: de", pixels.min().min(), "a", pixels.max().max())
treino_X = pixels / 255
treino_y = treino["digito"]
teste_X = teste.drop(columns="digito") / 255
teste_y = teste["digito"]
print("depois de dividir: de", numero_br(treino_X.min().min(), 0),
      "a", numero_br(treino_X.max().max(), 0))
contagem = treino_y.value_counts().sort_index()
print("imagens por dígito no treino:")
print("  menos:", contagem.idxmin(), f"({contagem.min()})")
print("  mais:", contagem.idxmax(), f"({contagem.max()})")

> **🔧 Função**
>
> - **`df.shape`** — o número de linhas e de colunas da tabela.
> - **`df.drop(columns="digito")`** — a tabela sem a coluna dada.
> - **`df.min().min()`** — o primeiro `min()` dá o menor valor de cada coluna, e o segundo, o menor entre eles; o mesmo com `max()`.
> - **`df / 255`** — divide todos os valores da tabela por 255.
> - **`serie.value_counts()`** — quantas vezes aparece cada valor; **`.sort_index()`** ordena o resultado pelo valor, aqui de 0 a 9.
> - **`serie.idxmin()`**, **`serie.idxmax()`** — o rótulo da posição com o menor e com o maior valor, aqui o dígito.

Cada tabela tem 785 colunas: o rótulo e os 784 pixels. No treino, o dígito com menos imagens é o 5, com 514, e o que tem mais é o 1, com 671; os dez dígitos aparecem em quantidades da mesma ordem. A figura mostra a primeira imagem de cada dígito no treino, com os 784 pixels de volta à grade de 28 × 28:

In [ ]:
# Figura: A primeira imagem de cada dígito, de 0 a 9, na tabela de treino. Cada quadro tem 28 × 28 pixels; quanto maior o valor do pixel, mais opaco o azul, e os pixels de valor 0 ficam transparentes.
tinta = LinearSegmentedColormap.from_list(
    "tinta", [to_rgba("C0", 0), to_rgba("C0", 1)])
fig, eixos = plt.subplots(2, 5, figsize=(3.8, 2.0))
for digito, ax in enumerate(eixos.ravel()):
    linha = (treino_y == digito).idxmax()
    imagem = treino_X.loc[linha].to_numpy().reshape(28, 28)
    ax.imshow(imagem, cmap=tinta, vmin=0, vmax=1)
    ax.set_title(str(digito))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for lado in ax.spines.values():
        lado.set_visible(True)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`LinearSegmentedColormap.from_list(nome, [cor0, cor1])`** — uma escala de cores que vai de `cor0`, no valor mínimo, a `cor1`, no máximo. **`to_rgba("C0", opacidade)`** dá o azul `C0` do estilo com a opacidade dada, de 0 (transparente) a 1.
> - **`(treino_y == digito).idxmax()`** — o rótulo da primeira linha em que a comparação dá `True`, isto é, a primeira imagem do dígito; **`df.loc[linha]`** é a linha da tabela com esse rótulo.
> - **`serie.to_numpy().reshape(28, 28)`** — os 784 valores numa grade de 28 linhas por 28 colunas, na ordem em que foram lidos.
> - **`ax.imshow(grade, cmap, vmin, vmax)`** — desenha a grade como imagem, um quadradinho por valor, com a cor dada pela escala `cmap` entre `vmin` e `vmax`.
> - **`ax.set_xticks([])`**, **`ax.set_yticks([])`**, **`ax.grid(False)`** — tiram as marcas dos eixos e a malha; **`ax.spines`** são as quatro bordas, que `set_visible(True)` mostra, para desenhar a moldura.

Para o modelo, cada imagem é um vetor $X = (X_1, \dots, X_p)$, com $p = 784$, e a resposta $Y$, o dígito, é qualitativa, com dez classes de 0 a 9. Ela entra no ajuste como dez variáveis indicadoras $Y_0, Y_1, \dots, Y_9$: $Y_m$ vale 1 se $Y = m$ e 0 se não, de modo que cada imagem tem um único 1. Essa representação é chamada de ***one-hot***. O `MLPClassifier`, a rede do `scikit-learn` para classificação, faz a conversão sozinho a partir dos rótulos; o código abaixo só a mostra, para as quatro primeiras imagens do treino:

In [ ]:
quatro = treino_y.iloc[:4].to_numpy()
indicadoras = np.eye(10, dtype=int)[quatro]
print(pd.DataFrame(indicadoras, index=quatro).to_string())

> **🔧 Função**
>
> **`np.eye(10, dtype=int)`** — a matriz identidade 10 × 10, com 1 na diagonal e 0 fora dela; a linha $m$ tem um único 1, na coluna $m$. **`matriz[quatro]`** — as linhas da matriz nas posições dadas por `quatro`, uma por imagem.

Cada linha é uma imagem, com o dígito à esquerda, e o único 1 da linha fica na coluna do dígito.

### Duas camadas escondidas

Uma rede de camada única, como a da seção 14.1, com unidades suficientes, consegue aproximar quase qualquer função dos preditores; esse é um resultado teórico, que não diz quantas unidades seriam necessárias nem como achar os pesos. Na prática, achar bons pesos costuma ser mais fácil com várias camadas de tamanho moderado, em que cada camada recebe as ativações da anterior (James et al., 2023). A rede abaixo tem duas camadas escondidas, $L_1$ com $K_1 = 256$ unidades e $L_2$ com $K_2 = 128$, e dez saídas, uma por dígito:

In [ ]:
# Figura: A rede de duas camadas escondidas para os dígitos: a entrada (cinza), as camadas $L_1$ e $L_2$ (azul) e as dez saídas (laranja), ligadas pelos pesos $w^{(1)}$, $w^{(2)}$ e $\beta$. Cada camada aparece com poucos círculos; o número embaixo é o de unidades de verdade.
colunas = [
    (0, [1.5, 0.5, -0.5, -1.5], "0.45", "entrada", "784"),
    (1, [1.5, 0.5, -0.5, -1.5], "C0", "$L_1$", "256"),
    (2, [1, 0, -1], "C0", "$L_2$", "128"),
    (3, [1, 0, -1], "C1", "saída", "10"),
]
ligacao = dict(arrowstyle="-", color="0.45", linewidth=0.4,
               alpha=0.7, shrinkA=7, shrinkB=7)
fig, ax = plt.subplots(figsize=(3.8, 3.0))
for (x0, ys0, *_), (x1, ys1, *_) in zip(colunas[:-1], colunas[1:]):
    for y0 in ys0:
        for y1 in ys1:
            ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=ligacao)
for x, ys, cor, nome, unidades in colunas:
    ax.scatter([x] * len(ys), ys, s=160, facecolors="none",
               edgecolors=cor, linewidths=1.5)
    ax.text(x, min(ys) - 0.75, "⋮", ha="center", va="center")
    ax.text(x, 2.35, nome, ha="center")
    ax.text(x, -2.85, unidades, ha="center")
for x, peso in [(0.5, "$w^{(1)}$"), (1.5, "$w^{(2)}$"),
                 (2.5, r"$\beta$")]:
    ax.text(x, 1.95, peso, ha="center")
ax.set_xlim(-0.4, 3.4)
ax.set_ylim(-3.1, 2.8)
ax.axis("off")
plt.tight_layout()
plt.show()

> **🔷 Conceito**
>
> Numa ***rede multicamada*** com duas camadas escondidas, a primeira camada calcula, para $k = 1, \dots, K_1$,
> $$
> A^{(1)}_k = g\Big(w^{(1)}_{k0} + \sum_{j=1}^{p} w^{(1)}_{kj} X_j\Big),
> $$
> com $g$ a função de ativação, a ReLU da seção 14.1. A segunda camada faz o mesmo com as ativações da primeira no lugar dos pixels, para $\ell = 1, \dots, K_2$:
> $$
> A^{(2)}_\ell = g\Big(w^{(2)}_{\ell 0} + \sum_{k=1}^{K_1} w^{(2)}_{\ell k} A^{(1)}_k\Big).
> $$
> O índice entre parênteses diz a camada.

As ativações $A^{(2)}_\ell$ da segunda camada são funções das ativações da primeira, que são funções dos pixels; logo, as da segunda também são funções dos pixels, por uma cadeia de transformações. A saída trata essas $K_2$ variáveis montadas pela rede como preditores:

> **🔷 Conceito**
>
> Para cada classe $m = 0, 1, \dots, 9$, a ***camada de saída*** calcula uma combinação linear
> $$
> Z_m = \beta_{m0} + \sum_{\ell=1}^{K_2} \beta_{m\ell} A^{(2)}_\ell,
> $$
> e a função ***softmax*** leva os dez $Z_m$ a probabilidades:
> $$
> \begin{aligned}
> f_m(X) &= \Pr(Y = m \mid X)\\
> &= \frac{e^{Z_m}}{\sum_{r=0}^{9} e^{Z_r}}.
> \end{aligned}
> $$
> A rede classifica a imagem no dígito de maior probabilidade.

As dez probabilidades são positivas e somam 1, qualquer que seja o valor dos $Z_m$. A softmax é a mesma função da logística multinomial da seção 9.3, em que cada classe tinha a sua combinação linear dos preditores. A diferença está no que entra na combinação linear: lá, os próprios preditores; aqui, as ativações $A^{(2)}_\ell$. Sem camadas escondidas, com os $Z_m$ calculados direto dos pixels, a rede seria a logística multinomial.

O `MLPClassifier` ajusta a rede das duas camadas às 6.000 imagens de treino, com a ReLU como ativação $g$. Os tamanhos 256 e 128 e a semente 14 são valores fixos, não ajustados; escolher o tamanho de uma rede sem olhar o teste é assunto da seção 14.4.

In [ ]:
rede = MLPClassifier(hidden_layer_sizes=(256, 128),
                     random_state=14)
rede.fit(treino_X, treino_y)
print("épocas:", rede.n_iter_)
print("saída:", rede.out_activation_)
for camada, (W, b) in enumerate(
        zip(rede.coefs_, rede.intercepts_), start=1):
    print(f"camada {camada}: {W.shape},",
          numero_br(W.size + b.size, 0), "pesos")
pesos = (sum(W.size for W in rede.coefs_)
         + sum(b.size for b in rede.intercepts_))
print("pesos da rede:", numero_br(pesos, 0))

> **🔧 Função**
>
> **`MLPClassifier(hidden_layer_sizes, random_state)`** — a rede neural de classificação do `scikit-learn`. `hidden_layer_sizes=(256, 128)` pede duas camadas escondidas, com 256 e 128 unidades; a ativação padrão é a ReLU. Os pesos iniciais são sorteados, e as imagens são embaralhadas a cada passada pelo treino; `random_state` fixa os dois sorteios. **`rede.n_iter_`** — quantas vezes o ajuste percorreu todas as imagens de treino, as ***épocas*** da seção 14.3. O ajuste para sozinho quando a perda de treino passa 10 épocas seguidas sem cair mais que uma pequena tolerância, ou quando chega ao limite `max_iter`, de 200 épocas no padrão. **`rede.out_activation_`** — a função aplicada na saída: `"softmax"` quando há mais de duas classes.

O ajuste parou em 47 épocas, antes do limite de 200, e a saída é a softmax. Todos os números da rede, os $w$ e os $\beta$, são chamados de ***pesos***, e os interceptos, como $w^{(1)}_{k0}$ e $\beta_{m0}$, de ***vieses*** (*biases*); esse viés não tem relação com o da decomposição em viés e variância da seção 7.6. Na figura, $w^{(1)}$ reúne todos os $w^{(1)}_{kj}$ da camada 1, vieses incluídos, $w^{(2)}$ todos os $w^{(2)}_{\ell k}$ e $\beta$ todos os $\beta_{m\ell}$. `coefs_[0]` é a tabela dos $w^{(1)}_{kj}$, com uma linha por pixel e uma coluna por unidade de $L_1$, e a camada 1 soma a esses $784 \times 256$ pesos os seus 256 vieses: $784 \times 256 + 256 = 785 \times 256 = 200.960$. Do mesmo jeito, a camada 2 tem $(256 + 1) \times 128 = 32.896$, e a de saída, $(128 + 1) \times 10 = 1.290$. O total é $200.960 + 32.896 + 1.290 = 235.146$ pesos.

Para cada imagem, a rede devolve dez probabilidades, uma por dígito. O código mostra as da primeira imagem de teste e as da primeira imagem de teste que a rede classifica errado:

In [ ]:
proba = rede.predict_proba(teste_X)
print("probabilidades:", proba.shape)
previsto = rede.predict(teste_X)
errada = int((previsto != teste_y).to_numpy().argmax())
duas = [0, errada]
print("imagens de teste:", [i + 1 for i in duas])
print("dígito certo:", teste_y.iloc[duas].tolist())
print("dígito previsto:", previsto[duas].tolist())
tabela = pd.DataFrame(
    {f"{i + 1}ª": proba[i] for i in duas},
    index=rede.classes_)
print(tabela.map(lambda p: numero_br(p, 3)).to_string())
print("todas somam 1:", np.allclose(proba.sum(axis=1), 1))
print("predict = a maior:",
      (proba.argmax(axis=1) == previsto).all())

> **🔧 Função**
>
> **`rede.predict_proba(X)`** — as probabilidades de cada classe, uma linha por imagem e uma coluna por classe, na ordem de `rede.classes_` (aqui, de 0 a 9). **`(previsto != teste_y).to_numpy().argmax()`** — a posição do primeiro `True`, isto é, a primeira imagem classificada errado; `int` a escreve como número comum. As posições começam em 0, e por isso o código soma 1 ao escrevê-las. **`serie.iloc[posicoes]`** — os valores nas posições dadas; **`.tolist()`** os escreve como lista. **`df.map(f)`** — aplica `f` a cada valor da tabela, aqui para escrevê-lo com a vírgula. **`proba.sum(axis=1)`** — a soma de cada linha. **`proba.argmax(axis=1)`** — a posição do maior valor de cada linha; como as classes vão de 0 a 9, a posição é o próprio dígito. **`(a == b).all()`** — `True` se as duas sequências são iguais em todas as posições.

A primeira imagem de teste é um 7, e a rede dá ao 7 probabilidade 1,000 e aos outros nove dígitos 0,000, em três casas. A 9ª imagem de teste é a primeira que a rede erra: é um 5, e a rede reparte quase toda a probabilidade entre cinco dígitos, com 0,891 no 6, 0,065 no 4, 0,022 no 5, 0,018 no 0 e 0,003 no 2; o 6 tem a maior, e é ele o previsto. As 2.000 linhas somam 1, e o `predict` é sempre o dígito de maior probabilidade.

#### A perda que o ajuste minimiza

Os pesos são escolhidos para que a rede dê probabilidade alta ao dígito certo de cada imagem de treino. Com $y_{im}$ igual a 1 se a imagem $i$ é do dígito $m$ e 0 se não é, a medida é a ***entropia cruzada***
$$
-\sum_{i=1}^{n} \sum_{m=0}^{9} y_{im} \log f_m(x_i).
$$
Em cada imagem, só a parcela do dígito certo tem $y_{im} = 1$, e a soma de dentro vale $-\log$ da probabilidade que a rede dá a esse dígito: próxima de 0 se a probabilidade é próxima de 1, e grande se ela é próxima de 0. Minimizar a entropia cruzada é maximizar a verossimilhança, o critério com que a regressão logística da seção 9.2 escolhia os coeficientes. O `MLPClassifier` minimiza a entropia cruzada média, a soma dividida por $n$, mais uma penalidade pequena sobre o tamanho dos pesos, controlada pelo parâmetro `alpha`.

In [ ]:
certa = proba[np.arange(len(teste_y)), teste_y]
for i in duas:
    print(f"imagem {i + 1}:",
          "p =", numero_br(certa[i], 3) + ",",
          "parcela =", numero_br(-np.log(certa[i]), 3))
media_treino = log_loss(treino_y, rede.predict_proba(treino_X))
print("média no treino:", numero_br(media_treino, 4))
print("alpha:", numero_br(rede.alpha, 4))
print("perda do ajuste:", numero_br(rede.loss_, 4))
print("perda acima da média:", rede.loss_ > media_treino)

> **🔧 Função**
>
> **`proba[np.arange(n), teste_y]`** — em cada linha $i$, a coluna do dígito certo daquela imagem: a probabilidade que a rede dá à resposta certa. **`log_loss(y, proba)`** — a entropia cruzada média, a soma de $-\log$ da probabilidade do dígito certo dividida pelo número de imagens. **`rede.alpha`** — o peso da penalidade sobre os pesos; o padrão é 0,0001. **`rede.loss_`** — a perda média da última época, calculada lote a lote enquanto os pesos ainda mudavam, com a penalidade incluída.

Na primeira imagem de teste, o 7 recebe probabilidade 1,000, e a parcela é 0,000. Na 9ª, o 5 recebe só 0,022, e a parcela é 3,811: é nas imagens em que a rede dá pouca probabilidade ao dígito certo que a entropia cruzada cresce. A entropia cruzada média nas 6.000 imagens de treino é 0,0006; a perda do ajuste, 0,0008, fica acima dela porque inclui a penalidade de `alpha` igual a 0,0001 e porque é uma média feita ao longo da última época, com os pesos ainda mudando.

#### Treino e teste

A rede tem 235.146 pesos para 6.000 imagens de treino. O que esperar do erro dela no treino? E no teste, em imagens que ela não viu? Faça um palpite antes da saída:

In [ ]:
erros_treino = (rede.predict(treino_X) != treino_y).sum()
erros_rede = (rede.predict(teste_X) != teste_y).sum()
print("pesos por imagem de treino:",
      numero_br(pesos / len(treino_y), 1))
print("erros no treino:", erros_treino, "de", len(treino_y))
print("erros no teste:", erros_rede, "de", len(teste_y))
print("entropia média no teste:",
      numero_br(log_loss(teste_y, proba), 3))

> **🔧 Função**
>
> **`(previsto != y).sum()`** — quantas previsões diferem do rótulo, isto é, o número de erros.

São 39,2 pesos por imagem de treino, e a rede não erra nenhuma das 6.000. No teste, erra 127 das 2.000, e a entropia cruzada média sobe para 0,298. Com tantos pesos, a rede pode se ajustar a detalhes das imagens de treino que não se repetem nas de teste. A penalidade `alpha`, pequena no padrão, e outras formas de conter o ajuste são assunto da seção 14.3.

### Contra os modelos lineares

Dois modelos do capítulo 9 separam as classes com fronteiras lineares nos pixels: a LDA, da seção 9.4, e a logística multinomial, da seção 9.3. A logística vai aqui com uma penalidade ridge nos coeficientes, como a da seção 11.3, controlada pelo parâmetro `C`. O valor `C=0.1` é fixo, não foi escolhido com o teste; a escolha por validação cruzada seria a da seção 11.5. A logística tem uma combinação linear dos 784 pixels por dígito. Antes de rodar o código, um palpite: ela erra mais ou menos que a rede nas 2.000 imagens de teste? E a LDA? E, no treino, qual dos três erra menos?

In [ ]:
lda = LinearDiscriminantAnalysis().fit(treino_X, treino_y)
logistica = LogisticRegression(C=0.1, max_iter=1000)
logistica.fit(treino_X, treino_y)
print("logística parou antes do limite:",
      logistica.n_iter_[0] < 1000)
print("pixels sempre 0 no treino:",
      (treino_X.max() == 0).sum())
print("coef_ da logística:", logistica.coef_.shape)
pesos_log = logistica.coef_.size + logistica.intercept_.size
print("pesos da logística:", numero_br(pesos_log, 0))
modelos = {"LDA": lda, "logística": logistica, "rede": rede}
conta = lambda m, X, y: (m.predict(X) != y).sum()
erros = pd.DataFrame({
    "teste": {nome: conta(m, teste_X, teste_y)
              for nome, m in modelos.items()},
    "treino": {nome: conta(m, treino_X, treino_y)
               for nome, m in modelos.items()},
})
taxas = erros / pd.Series({"teste": len(teste_y),
                           "treino": len(treino_y)})
print("erros:")
print(erros.to_string())
print("taxas de erro:")
print(taxas.map(lambda v: numero_br(v, 3)).to_string())
e = erros["teste"]
print("rede < logística < LDA:",
      e["rede"] < e["logística"] < e["LDA"])
print("rede / logística:", numero_br(e["rede"] / e["logística"]))
print("treino abaixo do teste:",
      (taxas["treino"] < taxas["teste"]).all())

> **🔧 Função**
>
> **`LinearDiscriminantAnalysis()`** — a LDA da seção 9.4. **`LogisticRegression(C, max_iter)`** — a regressão logística; com mais de duas classes, a multinomial, na forma softmax. `C` é o inverso da força da penalidade ridge sobre os coeficientes: quanto menor o `C`, mais os coeficientes são puxados para zero; o padrão é `C=1`. `max_iter` é o número máximo de passos do método que procura os coeficientes (o padrão é 100); os 1.000 daqui são uma folga, para que o ajuste pare por convergência e não pelo limite, e **`logistica.n_iter_`** diz quantos passos ele deu. **`logistica.coef_`** — os coeficientes, uma linha por classe e uma coluna por preditor; **`logistica.intercept_`**, um intercepto por classe. **`pd.DataFrame({coluna: {linha: valor}})`** — uma tabela a partir de dicionários, com as chaves de dentro como rótulos das linhas. **`pd.Series({rotulo: valor})`** — uma série a partir de um dicionário, com as chaves como rótulos. **`erros / pd.Series({coluna: n})`** — divide cada coluna da tabela pelo valor de mesmo nome na série. **`treino_X.max() == 0`** — `True` nas colunas cujo maior valor é 0, isto é, nos pixels que valem 0 em todas as imagens de treino; `.sum()` conta esses `True`.

A logística parou antes do limite de passos. Os pixels entram nela sem padronizar, ao contrário da regra da seção 11.3 para a ridge: a padronização põe na mesma escala colunas medidas em unidades diferentes, e aqui os 784 pixels já estão na mesma unidade, a intensidade de 0 a 1 (120 deles valem 0 em todas as imagens de treino). O mesmo vale para a penalidade `alpha` da rede, que também recebe os pixels de 0 a 1. A logística tem 10 linhas de 784 coeficientes e 10 interceptos, 7.850 pesos, uma combinação linear dos pixels por dígito.

No teste, a LDA erra 400 das 2.000 imagens, a logística 232 e a rede 127: a rede erra menos que a logística, e a logística menos que a LDA. A razão entre os erros da rede e os da logística é 0,55, pouco mais que a metade. No treino, a LDA erra 519 das 6.000 imagens, a logística 290 e a rede nenhuma, e nos três modelos a taxa de treino fica abaixo da de teste:

In [ ]:
# Figura: Taxa de erro no treino (azul, 6.000 imagens) e no teste (laranja, 2.000 imagens) da LDA, da logística multinomial com penalidade ridge e da rede de duas camadas escondidas. Uma divisão só, a das tabelas de treino e de teste.
ordem = ["rede", "logística", "LDA"]
posicoes = np.arange(len(ordem))
fig, ax = plt.subplots(figsize=(3.8, 2.8))
for coluna, cor, desvio in [("treino", "C0", 0.19),
                            ("teste", "C1", -0.19)]:
    valores = taxas.loc[ordem, coluna]
    ax.barh(posicoes + desvio, valores, height=0.36,
            color=cor, label=coluna)
    for y, valor in zip(posicoes + desvio, valores):
        ax.text(valor + 0.004, y, numero_br(valor, 3),
                va="center", fontsize=8)
ax.set_yticks(posicoes, ordem)
ax.set_xlim(0, 0.25)
ax.set_xlabel("taxa de erro")
ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(
    lambda v, _: numero_br(v, 2))
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.barh(posicoes, valores, height, color, label)`** — uma barra horizontal em cada posição vertical, com o comprimento dado pelo valor; `height` é a espessura da barra. Deslocar as posições para cima e para baixo põe as duas barras de cada modelo lado a lado. **`ax.set_yticks(posicoes, rotulos)`** — escreve os nomes dos modelos nas posições dadas. **`df.loc[linhas, coluna]`** — os valores da coluna nas linhas dadas, na ordem pedida. **`ax.grid(axis="y", visible=False)`** — tira só as linhas da malha que acompanham o eixo vertical.

A pergunta do começo tem resposta nestes dados: a rede de duas camadas escondidas erra menos imagens de teste que os dois modelos lineares, a LDA e a logística com `C=0.1`. A rede não recebeu nenhuma variável além dos pixels: a saída dela combina 128 variáveis que as duas camadas montaram a partir deles, com pesos estimados nas imagens de treino. A diferença medida é a destes três ajustes, nesta divisão (6.000 imagens de treino e 2.000 de teste), com `C` fixo para a logística e uma semente para a rede. Outra semente sorteia outros pesos iniciais e outra ordem das imagens, e, como outro `C`, pode mudar as contagens; a seção 14.3 volta a isso.

## Ajustando uma Rede: Retropropagação, Gradiente Estocástico e Regularização

> **📌 Nota**
>
> Esta seção corresponde à seção 10.7 de James et al. (2023).

Uma rede de cinco unidades ReLU liga os anos de carreira ao log do salário dos jogadores de beisebol, e o ajuste dela começa por pesos sorteados. Se o sorteio muda, sai a mesma curva? Se o ajuste pode terminar em lugares diferentes conforme o ponto de partida, como ele anda de um ponto a outro, e quando deve parar?

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Um problema com vários mínimos

O código abaixo lê os jogadores com salário e ajusta a mesma rede, cinco unidades ReLU sobre os anos padronizados, com o mesmo método de busca dos pesos, `solver="lbfgs"`, dez vezes. Só muda a semente (`random_state` de 0 a 9), e com ela os pesos sorteados no começo. Para cada ajuste, imprime o MSE de treino e confere se o `lbfgs` parou antes do limite de 5.000 iterações, isto é, porque o MSE deixou de cair e não porque as iterações acabaram:

In [ ]:
hitters = pd.read_csv("dados/Hitters.csv").dropna()
print("jogadores:", len(hitters))
y = np.log(hitters["salario"])
X = hitters[["anos"]]
grade = pd.DataFrame({"anos": np.linspace(1, 24, 300)})
mse, curvas, redes, iteracoes = {}, {}, {}, {}
for semente in range(10):
    rede = make_pipeline(
        StandardScaler(),
        MLPRegressor(hidden_layer_sizes=(5,), solver="lbfgs",
                     max_iter=5000, random_state=semente))
    rede.fit(X, y)
    mse[semente] = mean_squared_error(y, rede.predict(X))
    curvas[semente] = rede.predict(grade)
    redes[semente] = rede
    iteracoes[semente] = rede[-1].n_iter_
mse = pd.Series(mse).round(4)
for semente, valor in mse.items():
    print(f"semente {semente}: MSE {numero_br(valor, 4)}")
print("valores distintos:", mse.nunique())
print("de", numero_br(mse.min(), 4), "a", numero_br(mse.max(), 4))
comum = mse.value_counts()
print("o mais repetido:", numero_br(comum.index[0], 4),
      f"({comum.iloc[0]} sementes)")
print("todas pararam antes de 5.000:",
      max(iteracoes.values()) < 5000)

> **🔧 Função**
>
> **`serie.round(4)`** — cada valor arredondado a quatro casas. **`serie.nunique()`** — quantos valores diferentes a série tem. Em **`comum.index[0]`** e **`comum.iloc[0]`**, o resultado de `value_counts()` vem do valor mais frequente para o menos frequente: o primeiro rótulo é o MSE mais repetido, e o primeiro valor, quantas sementes chegaram a ele. Com `solver="lbfgs"`, **`.n_iter_`** conta as iterações do método, e não épocas.

Os 263 jogadores e as dez sementes dão seis valores diferentes de MSE de treino, de 0,3868 a 0,3958. O mais repetido, 0,3953, sai de cinco sementes. Nenhum ajuste esgotou as 5.000 iterações: todos pararam porque o MSE deixou de cair. Como a rede e os dados não mudam, a diferença vem só do ponto de partida. A figura mostra três das curvas: a da semente de menor MSE, a da primeira semente com o MSE mais repetido, e a da semente de maior MSE.

In [ ]:
# Figura: Log do salário contra os anos de carreira e a curva ajustada pela mesma rede de cinco unidades ReLU com três sementes: a de menor MSE de treino (azul), a primeira com o MSE mais repetido (laranja) e a de maior (verde, tracejada). Em cima, todos os anos, com os jogadores em cinza, deslocados um pouco na horizontal, ao acaso, para os pontos de um mesmo ano não se sobreporem; embaixo, só as curvas, de 9 a 24 anos, com o eixo vertical ampliado.
repetida = mse.index[mse == comum.index[0]][0]
escolhidas = [mse.idxmin(), repetida, mse.idxmax()]
tres_anos = pd.DataFrame({"anos": [1, 9, 24]})
nove = grade["anos"] >= 9
for semente in escolhidas:
    f = redes[semente].predict(tres_anos)
    print(f"semente {semente}, f̂ em 1, 9, 24 anos:")
    print("  ", " · ".join(numero_br(v) for v in f))
    variacao = np.ptp(curvas[semente][nove])
    print("  variação de 9 a 24:", numero_br(variacao))
rng = np.random.default_rng(14)
tremor = rng.uniform(-0.25, 0.25, size=len(X))
fig, (ax, ax_z) = plt.subplots(2, 1, figsize=(3.8, 5.0))
ax.scatter(X["anos"] + tremor, y, s=10, color="0.45", alpha=0.4,
           linewidths=0)
for semente, cor, traco in zip(escolhidas, ["C0", "C1", "C2"],
                               ["-", "-", "--"]):
    for eixo in (ax, ax_z):
        eixo.plot(grade["anos"], curvas[semente], color=cor,
                  linestyle=traco, linewidth=1.8,
                  label=f"semente {semente}")
ax.set_xlim(0, 25)
ax_z.set_xlim(9, 24)
ax_z.set_ylim(6.2, 6.6)
for eixo in (ax, ax_z):
    eixo.set_xlabel("anos de carreira")
    eixo.set_ylabel("log do salário")
    eixo.yaxis.set_major_formatter(lambda v, _: f"{v:g}".replace(".", ","))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3,
          handlelength=1.5, columnspacing=1)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.ptp(v)`** — a diferença entre o maior e o menor valor de `v`. **`" · ".join(textos)`** — junta os textos numa linha só, separados por " · ". Em **`ax.legend`**, `handlelength` e `columnspacing` encurtam o traço de amostra e o espaço entre as colunas da legenda, para ela caber na largura da figura.

A semente 8, a de menor MSE, começa mais alta: prevê 4,71 para um jogador de 1 ano, contra 4,56 das outras duas. A semente 2, a de maior MSE, fica horizontal, em 6,44: dos 9 aos 24 anos, a variação impressa é 0,00. As outras duas terminam, aos 24 anos, abaixo do valor que tinham aos 9: a semente 8 vai de 6,47 a 6,32, e a semente 0, de 6,46 a 6,34. São três funções diferentes, com MSE de treino próximos.

Por que o ajuste não chega sempre ao mesmo lugar? Junte todos os pesos da rede, os $w$ e os $\beta$, num vetor só, $\theta$. O ajuste procura o $\theta$ que torna pequeno
$$
R(\theta) = \frac12 \sum_{i=1}^{n} \big(y_i - f_\theta(x_i)\big)^2,
$$
a metade do RSS, com $f_\theta$ a rede com os pesos $\theta$. O fator $\tfrac12$ não muda onde fica o mínimo e deixa as derivadas mais limpas. Na regressão linear, $R$ é uma tigela, com um único fundo. Na rede, os pesos entram uns dentro dos outros, $\beta_k$ multiplicando $g$ de uma combinação dos $w_{kj}$, e $R(\theta)$ pode ter vários vales. Um exemplo simples: trocar de lugar duas unidades, com os seus pesos, dá a mesma função com outro $\theta$, e então um mesmo fundo aparece em vários pontos. E há vales de fundos diferentes, como sugerem os seis valores de MSE.

> **🔷 Conceito**
>
> Uma função é ***convexa*** quando tem a forma de uma tigela, como a $R$ da regressão linear: o segmento de reta entre dois pontos quaisquer do gráfico nunca passa por baixo dele, e o fundo é um só. A $R(\theta)$ de uma rede é ***não convexa***: pode ter vários vales. O fundo de um vale, um ponto em que nenhum passo pequeno diminui $R$, é um ***mínimo local***; o mais baixo de todos é o ***mínimo global***. O ajuste de uma rede anda a partir de um ponto sorteado e tende a parar no fundo do vale em que caiu, que depende do ponto de partida e não precisa ser o global.

Não dá para desenhar $R(\theta)$ inteira, com um eixo por peso, mas dá para andar por ela em linha reta. O MSE de treino é $R$ vezes $2/n$ e tem os mesmos vales. O código abaixo toma os pesos ajustados de duas sementes, a de menor e a de maior MSE, $\theta_a$ e $\theta_b$, e mede o MSE de treino nos pesos
$$
\theta(t) = (1 - t)\,\theta_a + t\,\theta_b,
$$
com $t$ de $-0{,}2$ a $1{,}2$: em $t = 0$ ficam os pesos da primeira semente, em $t = 1$ os da segunda, e entre os dois, uma mistura deles. A rede não é ajustada de novo; só os pesos dela são trocados antes de cada previsão:

In [ ]:
# Figura: MSE de treino da rede dos anos de carreira nos pesos do caminho em linha reta entre os pesos ajustados de duas sementes: a de menor MSE, em t = 0 (azul), e a de maior, em t = 1 (verde).
s_a, s_b = escolhidas[0], escolhidas[-1]
rede_a, rede_b = redes[s_a][-1], redes[s_b][-1]
print("pesos da rede:", sum(W.size for W in rede_a.coefs_)
      + sum(v.size for v in rede_a.intercepts_))
mistura = copy.deepcopy(redes[s_a])
ts = np.linspace(-0.2, 1.2, 57).round(3)
caminho = []
for t in ts:
    mistura[-1].coefs_ = [(1 - t) * A + t * B for A, B
                          in zip(rede_a.coefs_, rede_b.coefs_)]
    mistura[-1].intercepts_ = [(1 - t) * A + t * B for A, B
                               in zip(rede_a.intercepts_,
                                      rede_b.intercepts_)]
    caminho.append(mean_squared_error(y, mistura.predict(X)))
caminho = pd.Series(caminho, index=ts)
print(f"t = 0 (semente {s_a}):", numero_br(caminho[0.0], 4))
print(f"t = 1 (semente {s_b}):", numero_br(caminho[1.0], 4))
entre = caminho[(ts > 0) & (ts < 1)]
print("maior MSE entre 0 e 1:", numero_br(entre.max(), 1))
print("  em t =", numero_br(entre.idxmax(), 2))
passo = 0.025
fundos = all(caminho[round(c + d, 3)] > caminho[c]
             for c in (0.0, 1.0) for d in (-passo, passo))
print("sobe dos dois lados de 0 e de 1:", fundos)
fig, ax = plt.subplots(figsize=(3.8, 3.0))
ax.plot(caminho.index, caminho, color="0.45", linewidth=1.8)
for t, semente, cor in [(0.0, s_a, "C0"), (1.0, s_b, "C2")]:
    ax.plot(t, caminho[t], "o", color=cor, markersize=8,
            label=f"semente {semente}")
ax.set_xlim(-0.25, 1.25)
ax.set_xlabel("t")
ax.set_ylabel("MSE de treino")
ax.set_ylim(bottom=0)
for eixo in (ax.xaxis, ax.yaxis):
    eixo.set_major_formatter(lambda v, _: f"{v:g}".replace(".", ",")
                             .replace("-", "−"))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`copy.deepcopy(objeto)`** — uma cópia independente, com os pesos ajustados junto, para que trocar `coefs_` e `intercepts_` da cópia mude o que o `predict` dela calcula sem mexer na rede original. **`.round(3)`** num vetor do `numpy` — arredonda a três casas, para que 0 e 1 fiquem exatos na grade de $t$.

Em $t = 0$ e $t = 1$, o MSE é o das duas sementes, 0,3868 e 0,3958. Entre elas, o caminho sobe até cerca de 3,3, perto de $t = 0{,}4$, e, a partir de cada extremo, o MSE sobe para os dois lados: em linha reta, não se passa de uma semente à outra sem subir um morro. Isso sozinho não separa os vales. A linha reta é só um caminho entre muitos, e entre duas cópias do mesmo fundo, com as unidades trocadas de lugar, ela também pode subir. O que mostra que as duas sementes pararam em fundos diferentes são os dois MSE, 0,3868 e 0,3958, e as curvas diferentes da figura anterior.

### Gradiente descendente e retropropagação

O ajuste anda pelos vales com o gradiente descendente do capítulo 5: a partir de um ponto, calcula para onde $R$ sobe mais depressa e dá um passo pequeno no sentido contrário. Uma ***derivada parcial*** mede quanto $R$ muda quando só um peso muda um pouco, com todos os outros parados. O ***gradiente*** de $R$ no ponto $\theta^m$ é o vetor de todas as derivadas parciais, uma por peso, calculadas no ponto atual:
$$
\nabla R(\theta^m) = \frac{\partial R(\theta)}{\partial \theta}\bigg|_{\theta = \theta^m}.
$$
O passo é
$$
\theta^{m+1} \leftarrow \theta^m - \rho\, \nabla R(\theta^m),
$$
em que $\rho$ é a ***taxa de aprendizado***, o tamanho do passo. Com $\rho$ pequeno o bastante, cada passo não aumenta $R$; com $\rho$ grande demais, os passos pulam o fundo do vale, como no capítulo 5. O processo repete até $R$ parar de cair.

$R$ é uma soma de parcelas, uma por jogador, e o gradiente de uma soma é a soma dos gradientes. Para a rede de uma camada, escreva $z_{ik} = w_{k0} + \sum_j w_{kj} x_{ij}$, a entrada da unidade $k$ no jogador $i$. A previsão para ele é
$$
f_\theta(x_i) = \beta_0 + \sum_{k=1}^{K} \beta_k\, g(z_{ik}),
$$
e a parcela dele em $R$ é
$$
R_i(\theta) = \frac12 \big(y_i - f_\theta(x_i)\big)^2.
$$

Comece pelas derivadas parciais de $R_i$ em relação aos pesos da saída, os $\beta_k$, que multiplicam $g(z_{ik})$ diretamente. A regra da cadeia dá
$$
\frac{\partial R_i}{\partial \beta_k} = -\big(y_i - f_\theta(x_i)\big)\, g(z_{ik}):
$$
o resíduo do jogador $i$, com o sinal trocado, vezes a ativação da unidade $k$. A unidade que ativa mais nesse jogador recebe uma parte maior do resíduo.

O código abaixo faz essa conta para o segundo jogador da tabela, na rede da semente de menor MSE: a previsão, o resíduo e, para cada unidade $k$, a ativação $g(z_{ik})$, calculada com os pesos de `coefs_` e `intercepts_`, e a derivada $\partial R_i / \partial \beta_k$:

In [ ]:
ajustada = redes[s_a]
jogador = X.iloc[[1]]
print("anos de carreira:", jogador["anos"].iloc[0])
f_i = ajustada.predict(jogador)[0]
residuo = y.iloc[1] - f_i
print("y:", numero_br(y.iloc[1], 3), "· f̂:", numero_br(f_i, 3))
print("resíduo:", numero_br(residuo))
W, v = ajustada[-1].coefs_, ajustada[-1].intercepts_
z = ajustada[0].transform(jogador) @ W[0] + v[0]
g_z = np.maximum(0, z).ravel()
print("confere com o predict:",
      np.isclose(g_z @ W[1].ravel() + v[1][0], f_i))
derivadas = pd.DataFrame(
    {"g(z)": g_z, "derivada": -residuo * g_z + 0.0},
    index=[f"unidade {k}" for k in range(1, 6)])
print(derivadas.map(numero_br).to_string())

> **🔧 Função**
>
> **`ajustada[0].transform(jogador)`** — aplica ao jogador a padronização do primeiro passo do pipeline, a mesma que o `predict` aplica antes da rede. **`z @ W[0]`** — multiplica a linha do jogador pela tabela de pesos das entradas: uma soma $\sum_j w_{kj} x_{ij}$ por unidade. **`np.isclose(a, b)`** — `True` se os dois números são iguais a menos de uma diferença muito pequena. **`+ 0.0`** — transforma em `0.0` o `-0.0` que sai de multiplicar zero por um número negativo, para a tabela não imprimir "−0,00".

O jogador tem 3 anos de carreira e log do salário 6,174; a rede prevê 5,037, e o resíduo, 6,174 − 5,037 = 1,137 (1,14 na saída, com duas casas), é positivo: a previsão ficou abaixo. As unidades 1 a 3 têm ativação 0 para ele, e a derivada em relação aos $\beta_k$ delas é 0. As unidades 4 e 5 ativam, com 0,60 e 0,94, e as derivadas são −0,68 e −1,07, negativas. O passo anda contra o gradiente: por conta deste jogador, $\beta_4$ e $\beta_5$ aumentam, o $\beta_5$ mais que o $\beta_4$, e a previsão para ele sobe na direção do valor observado.

Os pesos das unidades, os $w_{kj}$, estão um elo mais para dentro: mudar $w_{kj}$ muda $z_{ik}$, que muda $g(z_{ik})$, que, multiplicado por $\beta_k$, muda a previsão. A cadeia ganha esses elos:
$$
\begin{aligned}
\frac{\partial R_i}{\partial w_{kj}} = {}&-\big(y_i - f_\theta(x_i)\big)\\
&\cdot \beta_k\, g'(z_{ik})\, x_{ij}.
\end{aligned}
$$
O mesmo resíduo chega à unidade $k$ passando pelo peso $\beta_k$ e pela derivada $g'$, e dali a cada preditor $j$. Com a ReLU, $g'(z)$ vale 0 para $z$ negativo e 1 para $z$ positivo: uma unidade desligada para o jogador $i$ não recebe nada do resíduo dele, e os pesos dela não mudam por causa desse jogador.

> **🔷 Conceito**
>
> ***Retropropagação*** (*backpropagation*) é o cálculo do gradiente pela regra da cadeia, da saída para a entrada: o resíduo de cada observação é repartido entre os pesos da última camada e, por eles, levado de volta camada a camada até os pesos da primeira. Numa rede com mais camadas, a conta é a mesma, com mais elos na cadeia.

O gradiente tem uma derivada por peso. O código abaixo conta os pesos de uma rede com 784 entradas, duas camadas escondidas de 256 e 128 unidades e 10 saídas, a forma da rede dos dígitos adiante, e mostra dois padrões do `MLPClassifier`, o método de busca e o tamanho do passo:

In [ ]:
camadas = [784, 256, 128, 10]
pesos = sum((entra + 1) * sai
            for entra, sai in zip(camadas[:-1], camadas[1:]))
print("pesos de 784-256-128-10:", numero_br(pesos, 0))
padrao = MLPClassifier()
print("solver padrão:", padrao.solver)
print("learning_rate_init:", numero_br(padrao.learning_rate_init, 3))
regressor = MLPRegressor()
print("iguais no MLPRegressor:",
      regressor.solver == padrao.solver
      and regressor.learning_rate_init == padrao.learning_rate_init)

> **🔧 Função**
>
> **`(entra + 1) * sai`** — os pesos de uma camada: um por par de unidades das duas camadas, mais um viés por unidade de saída. **`MLPClassifier().solver`** e **`.learning_rate_init`** — os valores padrão desses dois parâmetros, lidos de uma rede ainda não ajustada.

Na rede dos anos de carreira, são 16 derivadas a cada passo; numa rede dessa forma, 235.146, e a retropropagação calcula todas numa passada de ida pela rede, para as previsões, e uma de volta, para as derivadas. No `scikit-learn`, $\rho$ é o parâmetro `learning_rate_init`, 0,001 no padrão. O método padrão do `MLPRegressor` e do `MLPClassifier`, `solver="adam"`, parte desse $\rho$ e ajusta o tamanho do passo de cada peso a partir dos gradientes dos passos anteriores, um dos otimizadores adaptativos da seção 5.4. O `solver="lbfgs"` da rede dos anos de carreira também anda por passos guiados pelo gradiente, calculado com todos os jogadores a cada passo.

### Lotes e épocas

Somar o gradiente de todas as $n$ observações a cada passo custa caro quando $n$ é grande. A alternativa é calcular cada passo com uma amostra pequena das observações, um ***lote*** (*minibatch*): o gradiente do lote aponta mais ou menos para onde aponta o gradiente de todas, e sai muito mais barato. É o gradiente por *minibatch* da seção 5.6. Em redes neurais, esse método leva o nome de ***gradiente descendente estocástico*** (SGD, de *stochastic gradient descent*) qualquer que seja o tamanho do lote, e não só com uma observação por passo, que é o método que a seção 5.6 chama de estocástico. Uma ***época*** é uma passada por todas as observações de treino: com lotes de $b$ observações, uma época tem cerca de $n/b$ passos.

Para ver o que acontece ao longo das épocas, é preciso medir, depois de cada uma, o erro em imagens que não entraram no ajuste. O próximo trecho lê as imagens de treino e de teste do MNIST e separa das de treino um quinto para ***validação***, com a mesma proporção de cada dígito; as outras ficam para o ajuste dos pesos. O lote padrão do `MLPClassifier` tem $b = \min(200, n)$ imagens:

In [ ]:
treino = pd.read_csv("dados/mnist_treino.csv.gz")
teste = pd.read_csv("dados/mnist_teste.csv.gz")
treino_X = treino.drop(columns="digito") / 255
treino_y = treino["digito"]
teste_X = teste.drop(columns="digito") / 255
teste_y = teste["digito"]
print("treino:", numero_br(len(treino), 0),
      "· teste:", numero_br(len(teste), 0))
ajuste_X, valid_X, ajuste_y, valid_y = train_test_split(
    treino_X, treino_y, test_size=0.2, random_state=14,
    stratify=treino_y)
print("ajuste:", numero_br(len(ajuste_X), 0), "imagens")
print("validação:", numero_br(len(valid_X), 0), "imagens")
lote = min(200, len(ajuste_X))
print("lote:", lote, "imagens")
print("passos por época:", len(ajuste_X) // lote)

> **🔧 Função**
>
> **`train_test_split(X, y, test_size, random_state, stratify)`** — separa as linhas em duas partes sorteadas: `test_size=0.2` põe um quinto na segunda parte, `random_state` fixa o sorteio e `stratify=treino_y` mantém em cada parte a proporção de cada dígito. Devolve as duas partes de `X` e depois as duas de `y`.

Das 6.000 imagens de treino, 4.800 ficam para o ajuste e 1.200 para a validação. Com lotes de 200, uma época são $4.800 / 200 = 24$ passos de gradiente.

O ajuste abaixo avança uma época por vez, com `partial_fit`, e depois de cada época guarda as probabilidades que a rede dá a cada dígito nas imagens de ajuste, nas de validação e nas 2.000 de teste. A rede é a de duas camadas escondidas, com 256 e 128 unidades. O laço fica numa função, `treina`, porque vai ser repetido mais adiante com outra penalidade e com outras sementes:

In [ ]:
def treina(alpha, semente=14, epocas=30):
    rede = MLPClassifier(hidden_layer_sizes=(256, 128), alpha=alpha,
                         random_state=np.random.RandomState(semente))
    probs = []
    for epoca in range(epocas):
        rede.partial_fit(ajuste_X, ajuste_y, classes=np.arange(10))
        probs.append([rede.predict_proba(Z)
                      for Z in (ajuste_X, valid_X, teste_X)])
    return rede, probs

rede, probs = treina(alpha=1e-4)
print("épocas:", len(probs))
print("passos:", len(probs) * len(ajuste_X) // lote)
print("imagens processadas (t_):", numero_br(rede.t_, 0))

> **🔧 Função**
>
> **`rede.partial_fit(X, y, classes)`** — faz **uma** época de ajuste a partir dos pesos atuais, sem recomeçar; na primeira chamada, `classes` lista todas as classes possíveis. **`np.random.RandomState(14)`** — um gerador de números aleatórios com semente 14. Passado como `random_state`, ele continua de uma chamada de `partial_fit` para a outra, e cada época embaralha as imagens de um jeito. Com o número 14 no lugar dele, cada chamada recomeçaria o sorteio do zero; a primeira gasta parte dele nos pesos iniciais, e da segunda época em diante o embaralhamento seria sempre o mesmo. **`rede.t_`** — o número de imagens processadas no ajuste, somado ao longo das épocas.

Foram 30 épocas, $30 \times 24 = 720$ passos de gradiente, e cada uma das 4.800 imagens de ajuste passou 30 vezes pela rede: $30 \times 4.800 = 144.000$ imagens processadas. Das probabilidades de cada época saem a entropia cruzada média e o número de erros em cada parte das imagens:

In [ ]:
partes = {"ajuste": ajuste_y, "valid": valid_y, "teste": teste_y}

def mede(probs):
    linhas = []
    for p_epoca in probs:
        linha = {}
        for (nome, alvo), p in zip(partes.items(), p_epoca):
            linha[f"perda_{nome}"] = log_loss(alvo, p)
            linha[f"erros_{nome}"] = (p.argmax(axis=1) != alvo).sum()
        linhas.append(linha)
    return pd.DataFrame(linhas, index=range(1, len(probs) + 1))

curva = mede(probs)
print("épocas:", len(curva))
print("colunas:", len(curva.columns))

> **🔧 Função**
>
> **`zip(partes.items(), p_epoca)`** — anda junto pelos pares (nome, respostas certas) e pelas três tabelas de probabilidades da época. **`pd.DataFrame(linhas, index=...)`** — uma tabela com uma linha por dicionário, com as épocas, de 1 a 30, como rótulos das linhas.

Depois de 30 épocas, a entropia cruzada nas imagens de validação continua caindo, como nas de ajuste? E a taxa de erro? Faça o seu palpite antes de olhar as curvas:

In [ ]:
# Figura: Entropia cruzada média (em cima) e taxa de erro (embaixo) da rede de duas camadas escondidas depois de cada época, nas 4.800 imagens de ajuste (azul) e nas 1.200 de validação (laranja).
fig, (ax_p, ax_e) = plt.subplots(2, 1, figsize=(3.8, 4.6), sharex=True)
for coluna, nome, cor in [("ajuste", "ajuste", "C0"),
                          ("valid", "validação", "C1")]:
    ax_p.plot(curva.index, curva[f"perda_{coluna}"], color=cor,
              label=nome)
    ax_e.plot(curva.index,
              curva[f"erros_{coluna}"] / len(partes[coluna]), color=cor)
ax_p.set_ylabel("entropia cruzada")
ax_e.set_ylabel("taxa de erro")
ax_e.set_xlabel("época")
ax_e.set_xlim(0, 31)
ax_p.set_ylim(bottom=0)
ax_e.set_ylim(bottom=0)
for ax in (ax_p, ax_e):
    ax.yaxis.set_major_formatter(lambda v, _: numero_br(v, 2))
ax_p.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.set_ylim(bottom=0)`** — fixa só o limite de baixo do eixo vertical, aqui em zero. **`ax.set_xlim(0, 31)`** — os limites do eixo horizontal, que o `sharex=True` estende ao gráfico de cima.

Nas imagens de ajuste, as duas curvas descem até perto de zero. Nas de validação, a entropia cruzada desce nas primeiras épocas e depois para de descer. Os números:

In [ ]:
pv = curva["perda_valid"]
melhor = pv.idxmin()
print("menor perda de validação:", numero_br(pv.min(), 4))
print("  na época", melhor)
print("perda na época 30:", numero_br(pv[30], 4))
print("época 30 acima do mínimo:", pv[30] > pv.min())
pa = curva["perda_ajuste"]
print("perda de ajuste sempre cai:", (pa.diff().dropna() < 0).all())
print("perda de ajuste na 30:", numero_br(pa[30], 4))
print("erros de ajuste na 30:", curva.loc[30, "erros_ajuste"])
ev = curva["erros_valid"]
print("menos erros de validação:", ev.min(), "de",
      numero_br(len(valid_y), 0))
print("  nas épocas", ev.index[ev == ev.min()].tolist())
print("erros de validação na 30:", ev[30])

> **🔧 Função**
>
> **`serie.diff()`** — a diferença de cada valor para o anterior; o primeiro fica vazio, e **`.dropna()`** o tira. **`serie.index[condicao]`** — os rótulos das linhas em que a condição vale `True`. Em `curva`, os rótulos são as épocas, e `idxmin()` devolve a época de menor perda.

A entropia cruzada de validação chega ao menor valor, 0,2090, na época 10, e termina a época 30 em 0,2405, acima dele. Nas imagens de ajuste, ela cai a cada época, da primeira à 30ª, até 0,0023, e na época 30 a rede não erra nenhuma das 4.800. É o sobreajuste da seção 7.6, agora ao longo do tempo de ajuste: a rede continua melhorando nas imagens que vê e deixa de melhorar nas que não vê.

O menor número de erros de validação, 61 de 1.200, aparece nas épocas 23 e 27, bem depois da época 10, e na época 30 são 65. Como a perda pode subir enquanto os erros caem? A entropia cruzada média é a média, nas 1.200 imagens, da parcela $-\log$ da probabilidade dada ao dígito certo, e dá para separá-la em duas partes: a soma das parcelas das imagens erradas e a das acertadas, cada uma dividida por 1.200. O código abaixo faz isso nas épocas 10 e 30:

In [ ]:
def partes_da_perda(p_val):
    alvo = valid_y.to_numpy()
    parcela = -np.log(p_val[np.arange(len(alvo)), alvo])
    errou = p_val.argmax(axis=1) != alvo
    return (errou.sum(), parcela[errou].sum() / len(alvo),
            parcela[~errou].sum() / len(alvo))

for e in (melhor, 30):
    n_err, erradas, certas = partes_da_perda(probs[e - 1][1])
    print(f"época {e}: {n_err} erradas")
    print("  parte das erradas:", numero_br(erradas, 3))
    print("  parte das acertadas:", numero_br(certas, 3))

> **🔧 Função**
>
> **`~errou`** — troca `True` por `False` e vice-versa: as imagens acertadas. **`probs[e - 1][1]`** — as probabilidades de validação da época `e`; a lista começa na posição 0, que é a época 1.

Da época 10 para a 30, a parte das imagens erradas sobe de 0,164 para 0,221, e a das acertadas cai de 0,045 para 0,019. A rede passa a acertar algumas imagens a mais e fica mais segura nas que acerta, mas também nas que erra, e a entropia cruzada pune essa segurança: a subida da parte das erradas, $0{,}221 - 0{,}164 = 0{,}057$, é maior que a da perda inteira, $0{,}2405 - 0{,}2090 = 0{,}0315$.

Isso sugere parar o ajuste antes do fim: acompanhar a perda de validação a cada época e ficar com os pesos da época de menor perda. É a ***parada antecipada*** (*early stopping*), uma forma de regularização, porque impede que os pesos se ajustem demais às imagens de ajuste. O código abaixo compara, nas 2.000 imagens de teste, três épocas: a 10, de menor perda de validação; a primeira de menos erros de validação; e a 30:

In [ ]:
for e in (melhor, ev.idxmin(), 30):
    print(f"época {e}: perda de teste",
          numero_br(curva.loc[e, "perda_teste"], 4))
    print("  erros de teste:", curva.loc[e, "erros_teste"])

Com a semente 14, parar na época 10 dá, no teste, entropia cruzada 0,2748, contra 0,2929 na época 30: pela perda, a parada ganha. Mas a rede da época 10 erra 164 imagens de teste, contra 140 da época 30. Parar pelos erros de validação, na época 23, também não resolve aqui: 144 erros. O critério de parada tem de ser o que interessa: se o que se usa são as probabilidades, a perda de validação; se é o número de acertos, a taxa de erro. O `MLPClassifier` tem a parada antecipada como opção pronta, que aparece na seção 14.4.

### Regularização

A outra forma de conter o ajuste é a penalidade da ridge, da seção 11.3, sobre os pesos. Na classificação dos dígitos, com $y_{im}$ igual a 1 se a imagem $i$ é do dígito $m$, o ajuste minimiza
$$
\begin{aligned}
R(\theta; \lambda) = {}&-\sum_{i=1}^{n} \sum_{m=0}^{9} y_{im} \log f_m(x_i)\\
&+ \lambda \sum_j \theta_j^2,
\end{aligned}
$$
a entropia cruzada somada mais $\lambda$ vezes a soma dos quadrados dos pesos. Quanto maior o $\lambda$, mais os pesos são puxados para zero.

No `MLPClassifier`, a força da penalidade é o parâmetro `alpha`, e a correspondência com o $\lambda$ da fórmula é
$$
\lambda = \frac{\texttt{alpha}\cdot n}{2b},
$$
com $n$ imagens de ajuste e lotes de $b$ imagens. A rede ajustada até aqui usou o padrão, `alpha=1e-4`. O código abaixo calcula o $\lambda$ dos dois valores de `alpha` e repete o ajuste com `alpha=1`:

In [ ]:
for nome, a in [("0,0001", 1e-4), ("1", 1)]:
    print(f"alpha {nome}: λ =",
          numero_br(a * len(ajuste_X) / (2 * lote), 4))
rede_pen, probs_pen = treina(alpha=1)
curva_pen = mede(probs_pen)
nomes = ["alpha 0,0001", "alpha 1"]
tabela = pd.DataFrame(
    {nome: [c["perda_valid"].min(), c.loc[30, "perda_valid"]]
     for nome, c in zip(nomes, (curva, curva_pen))},
    index=["mínimo", "fim"])
tabela.loc["subida"] = tabela.loc["fim"] - tabela.loc["mínimo"]
tabela.loc["erradas"] = [partes_da_perda(p[-1][1])[1]
                         for p in (probs, probs_pen)]
print("perda de validação (fim: época 30):")
print(tabela.map(lambda v: numero_br(v, 4)).to_string())
contagens = pd.DataFrame(
    {nome: c.loc[30, ["erros_ajuste", "erros_valid", "erros_teste"]]
     .to_numpy() for nome, c in zip(nomes, (curva, curva_pen))},
    index=["ajuste", "validação", "teste"]).astype(int)
print("erros na época 30:")
print(contagens.to_string())
quadrados = [sum((W ** 2).sum() for W in r.coefs_)
             for r in (rede, rede_pen)]
print("soma dos quadrados dos pesos:")
print("  alpha 0,0001:", numero_br(quadrados[0], 0))
print("  alpha 1:", numero_br(quadrados[1], 0))

> **🔧 Função**
>
> **`MLPClassifier(alpha)`** — a força da penalidade sobre os pesos (sem os vieses); o padrão é `alpha=0.0001`. **`df.loc[nova] = valores`** — acrescenta uma linha com o rótulo `nova`. **`df.astype(int)`** — converte os valores para inteiros.

Com $n = 4.800$ e $b = 200$, `alpha=1` corresponde a $\lambda = 12$, e o padrão, a $\lambda = 0{,}0012$. A conta vem de como cada passo é dado: em cada lote de $b$ imagens, o passo reduz a entropia cruzada média do lote mais $\texttt{alpha}/(2b)$ vezes a soma dos quadrados dos pesos. Em média, ao longo dos lotes, isso é a entropia cruzada média das $n$ imagens de ajuste mais o mesmo termo, e multiplicar tudo por $n$ dá a fórmula acima, com $\lambda = \texttt{alpha}\cdot n / (2b)$. No `scikit-learn`, a soma dos quadrados inclui os pesos de todas as camadas, inclusive os da saída, mas não os vieses.

A soma dos quadrados dos pesos caiu de 750 para 94, e a rede penalizada ainda erra, na época 30, 41 das 4.800 imagens de ajuste. A entropia cruzada de validação terminou em 0,1990, só 0,0018 acima do menor valor dela, contra uma subida de 0,0315 sem a penalidade. A parte das imagens de validação erradas na perda média ficou em 0,1248, contra 0,2213: com os pesos contidos, a rede não chega a ficar tão segura nas imagens que erra.

Na época 30, a rede penalizada erra 66 das 1.200 imagens de validação, contra 65, e 160 das 2.000 de teste, contra 140. Com esta semente, a penalidade conteve a subida da perda, mas a rede penalizada erra mais imagens de teste. Com outras sementes, a comparação se mantém? O código abaixo repete os dois ajustes com as sementes de 0 a 4, com a mesma função `treina`, e guarda a perda de validação e os erros de teste da época 30:

In [ ]:
erros, perdas = {}, {}
for semente in range(5):
    for nome, a in zip(nomes, (1e-4, 1)):
        _, probs_s = treina(alpha=a, semente=semente)
        fim = mede(probs_s[-1:]).iloc[0]
        erros[(semente, nome)] = fim["erros_teste"]
        perdas[(semente, nome)] = fim["perda_valid"]
erros = pd.Series(erros).unstack().astype(int)
perdas = pd.Series(perdas).unstack()
erros.index.name = "semente"
print("erros de teste na época 30:")
print(erros.to_string())
mais = (erros["alpha 1"] > erros["alpha 0,0001"]).sum()
menor = (perdas["alpha 1"] < perdas["alpha 0,0001"]).sum()
print("alpha 1 com mais erros:", mais, "de 5")
print("alpha 1 com perda menor:", menor, "de 5")

> **🔧 Função**
>
> **`pd.Series(dicionario)`** com chaves que são pares (semente, nome) — uma série com rótulo de dois níveis; **`serie.unstack()`** passa o segundo nível, o `alpha`, para as colunas, e fica uma linha por semente. **`probs_s[-1:]`** — a lista só com a última época, para que `mede` meça só ela. **`df.index.name`** — o nome dos rótulos das linhas.

Com as sementes de 0 a 4 e a mesma divisão entre ajuste e validação, a rede com `alpha=1` termina com a perda de validação menor nas cinco e erra mais imagens de teste em quatro delas. Na semente 0, a rede penalizada erra uma imagem a menos, 162 contra 163. Nenhum dos dois valores de `alpha` foi escolhido; escolher `alpha` por validação cruzada, junto com o tamanho da rede, é o assunto da seção 14.4.

As curvas de validação das duas penalidades com a semente 14:

In [ ]:
# Figura: Entropia cruzada média nas 1.200 imagens de validação depois de cada época, com a penalidade padrão, alpha = 0,0001 (laranja), e com alpha = 1 (verde, tracejada).
fig, ax = plt.subplots(figsize=(3.8, 3.0))
ax.plot(curva.index, curva["perda_valid"], color="C1",
        label="alpha 0,0001")
ax.plot(curva_pen.index, curva_pen["perda_valid"], color="C2",
        linestyle="--", label="alpha 1")
ax.set_xlim(0, 31)
ax.set_xlabel("época")
ax.set_ylabel("entropia cruzada")
ax.set_ylim(bottom=0)
ax.yaxis.set_major_formatter(lambda v, _: numero_br(v, 2))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

### *Dropout*

Uma terceira forma de regularização sorteia, em vez de penalizar. A cada observação de treino processada, uma fração $\phi$ das unidades de uma camada é sorteada e desligada: as ativações delas viram zero para aquela observação, e as unidades que sobram têm os pesos multiplicados por $1/(1 - \phi)$, para compensar as que saíram. Com $\phi = 0{,}5$, por exemplo, numa camada de 128 unidades, cada imagem processada usa cerca de $128 \times 0{,}5 = 64$ delas, sorteadas de novo para a imagem seguinte, mesmo dentro do mesmo lote, e os pesos que saem das que ficaram valem $1/(1 - 0{,}5) = 2$ vezes o que valeriam: uma unidade que, com todas presentes, somaria 0,3 à entrada da camada seguinte soma 0,6. Numa previsão, todas as unidades participam, sem o fator.

> **🔷 Conceito**
>
> No ***dropout***, cada observação processada no ajuste passa por uma rede com uma parte das unidades desligada por sorteio. Nenhuma unidade pode contar com a presença de outra específica, e isso impede que algumas unidades se especializem demais nas observações de treino. É uma forma de regularização, semelhante em alguns aspectos à penalidade ridge.

A ideia é parente da floresta aleatória da seção 13.3, que sorteia as colunas que cada corte pode usar para que as árvores não dependam todas das mesmas colunas fortes. O *dropout* existe nas bibliotecas de aprendizado profundo, como uma camada a mais na definição da rede. O `MLPClassifier` e o `MLPRegressor` não o oferecem; as formas de regularização que eles têm são a penalidade `alpha` e a parada antecipada:

In [ ]:
for modelo in (MLPClassifier(), MLPRegressor()):
    print(type(modelo).__name__, "tem 'dropout':",
          any("dropout" in nome for nome in modelo.get_params()))

> **🔧 Função**
>
> **`modelo.get_params()`** — um dicionário com os parâmetros do modelo e os valores deles; percorrê-lo dá os nomes. **`type(modelo).__name__`** — o nome da classe do modelo. **`any(...)`** — `True` se algum dos valores for `True`.

### Onde o ajuste termina

A curva dos anos de carreira muda com a semente: a mesma rede, nos mesmos dados, terminou em seis MSE de treino diferentes em dez sementes, e o caminho em linha reta entre duas delas passa por um morro. A busca pelos pesos tende a parar no fundo do vale em que começou. Por isso, o MSE de treino não diz qual dessas redes prevê melhor jogadores novos: essa comparação se faz em dados que não entraram no ajuste, e as escolhas, como a época de parada e o `alpha`, se fazem na validação, deixando o teste para medir o resultado. O ajuste anda por passos contra o gradiente, calculado pela retropropagação, e, com lotes, dá muitos passos por época. Quando parar e quanto penalizar os pesos também se decide com imagens que não entraram no ajuste, e o critério tem de ser o que interessa: nos dígitos, com a semente 14, parar pela perda de validação deu probabilidades melhores no teste e mais erros. Com a semente 14, a penalidade forte conteve a subida da perda de validação; nas cinco sementes de 0 a 4, deu perda de validação menor e, em quatro delas, mais erros de teste.

## MLPClassifier e MLPRegressor na Prática

> **📌 Nota**
>
> Esta seção corresponde às seções 10.7.4 e 10.9.1 de James et al. (2023).

O salário de um jogador de beisebol, em milhares de dólares, depende das estatísticas dele: rebatidas na temporada, anos de carreira, *home runs* acumulados. São 19 colunas, e as escalas delas vão de indicadoras que valem 0 ou 1 a contagens de carreira na casa das dezenas de milhares. O que é preciso fazer com esses números antes de pedir a uma rede neural que preveja o salário? E, com a rede montada, como escolher quantas unidades ela tem e quanto penalizar os pesos, sem olhar os jogadores de teste?

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import (GridSearchCV, KFold, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}"
    .replace(",", "_").replace(".", ",")
    .replace("_", ".").replace("-", "−")
)

### Os jogadores e duas referências

A tabela `Hitters` traz as estatísticas de 1986 dos jogadores das grandes ligas e o salário de 1987, em milhares de dólares. Descartados os jogadores sem salário, as três colunas de texto (liga, divisão e liga na temporada seguinte) viram indicadoras, e um terço dos jogadores fica para teste. Em seguida, a distância entre as escalas, com o maior dos máximos das colunas e o menor deles, cada um com o nome da coluna:

In [ ]:
hitters = pd.read_csv(
    "dados/Hitters.csv").dropna()
y = hitters["salario"]
X = pd.get_dummies(
    hitters.drop(columns="salario"), drop_first=True, dtype=float)
print("jogadores:", len(X), "· colunas:", X.shape[1])
treino_X, teste_X, treino_y, teste_y = train_test_split(
    X, y, test_size=1/3, random_state=14)
print("treino:", len(treino_X), "· teste:", len(teste_X))
maximos = X.max()
print("maior valor:", numero_br(maximos.max(), 0))
print("  em", maximos.idxmax())
print("menor máximo:", numero_br(maximos.min(), 0))
print("  em", maximos.idxmin())

> **🔧 Função**
>
> **`pd.get_dummies(df, drop_first=True, dtype=float)`** — troca cada coluna de texto por colunas indicadoras, uma por categoria, com 1 na categoria da linha e 0 nas outras; `drop_first=True` deixa de fora a primeira categoria de cada coluna, em ordem alfabética, que fica representada por todas as indicadoras em 0, e `dtype=float` escreve as indicadoras como números.

São 263 jogadores e 19 colunas, com 175 jogadores no treino e 88 no teste. A coluna `vezes_ao_bastao_carreira` chega a 14.053; a indicadora `liga_Nacional` não passa de 1.

Duas previsões servem de referência. A mais simples ignora as colunas e prevê para todo jogador de teste o salário médio do treino. A outra é a regressão linear por mínimos quadrados com as 19 colunas. O erro é medido pelo ***erro absoluto médio*** (MAE, de *mean absolute error*),
$$
\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat y_i|,
$$
a distância média entre o salário e a previsão, em milhares de dólares, a mesma unidade do salário:

In [ ]:
media = treino_y.mean()
print("salário médio no treino:", numero_br(media, 0))
nulo = np.full(len(teste_y), media)
mae_nulo = mean_absolute_error(teste_y, nulo)
print("MAE da média:", numero_br(mae_nulo))
linear = LinearRegression().fit(treino_X, treino_y)
previsto = linear.predict(teste_X)
mae_linear = mean_absolute_error(teste_y, previsto)
print("MAE da linear:", numero_br(mae_linear))
print("R² da linear:", numero_br(r2_score(teste_y, previsto)))

> **🔧 Função**
>
> - **`np.full(n, valor)`** — um vetor com `n` cópias de `valor`.
> - **`mean_absolute_error(y, previsto)`** — o MAE, a média de $|y_i - \hat y_i|$.
> - **`r2_score(y, previsto)`** — o $R^2$, a fração da variação de `y` em torno da média que a previsão explica; é 1 com a previsão perfeita, 0 com a média de `y` como previsão, e negativo quando a previsão erra mais que essa média.

Prevendo a média, 557 mil dólares, o MAE de teste é de 344,05 mil dólares; a regressão linear o reduz a 232,06, com $R^2$ de teste de 0,48. Uma rede que erre mais que 344,05 está pior que ignorar as colunas.

### Escala: dos preditores e da resposta

Nas seções 14.1 e 14.3, a resposta era o log do salário. Aqui ela fica em milhares de dólares, para que o MAE se leia direto em dinheiro. Os preditores vão padronizados por um `StandardScaler`, como na seção 14.1, numa rede de uma camada escondida com 64 unidades, ajustada com o método padrão, o `adam`, e os demais padrões do `MLPRegressor`.

O ajuste pode terminar com um aviso, e o aviso diz algo sobre ele. A função `ajusta_e_avisa` ajusta o modelo guardando os avisos numa lista, em vez de mostrá-los, e devolve o tipo de cada um:

In [ ]:
from warnings import catch_warnings, simplefilter

def ajusta_e_avisa(modelo, X, y):
    with catch_warnings(record=True) as avisos:
        simplefilter("always")
        modelo.fit(X, y)
    return [a.category.__name__ for a in avisos]

> **🔧 Função**
>
> - **`catch_warnings(record=True)`**, do módulo `warnings` — num bloco `with`, guarda numa lista, aqui `avisos`, os avisos emitidos pelo código, em vez de mostrá-los; **`simplefilter("always")`** faz todo aviso entrar na lista. **`.category.__name__`** é o nome do tipo de cada aviso.

Antes de rodar, um palpite: com o salário em milhares de dólares, a rede erra mais ou menos que a média?

In [ ]:
so_X = Pipeline([
    ("escala", StandardScaler()),
    ("rede", MLPRegressor(
        hidden_layer_sizes=(64,),
        random_state=14)),
])
avisos = ajusta_e_avisa(so_X, treino_X, treino_y)
print("avisos:", len(avisos), *avisos)
print("lote:", min(200, len(treino_X)), "de", len(treino_X))
print("épocas:", so_X[-1].n_iter_)
perda = so_X[-1].loss_curve_
zero = 0.5 * (treino_y ** 2).mean()
print("perda na 1ª / a de prever 0:",
    numero_br(perda[0] / zero))
print("perda na 200ª / na 1ª:",
    numero_br(perda[-1] / perda[0]))
prev_so_X = so_X.predict(teste_X)
mae_so_X = mean_absolute_error(teste_y, prev_so_X)
print("MAE:", numero_br(mae_so_X))
print("pior que a média:", mae_so_X > mae_nulo)
print("maior previsão:", numero_br(prev_so_X.max(), 0))
print("mediana do teste:", numero_br(teste_y.median(), 1))

> **🔧 Função**
>
> - **`Pipeline([(nome, passo), ...])`** — encadeia os passos, como o `make_pipeline`, mas com um nome escolhido para cada um; os nomes servem para chegar a um passo (`so_X[-1]` é o último) e aos parâmetros dele.
> - **`*avisos`** — no `print`, passa cada item da lista como um valor separado; com a lista vazia, não imprime nada.
> - **`.loss_curve_`** — a perda de treino de cada época: metade do erro quadrático médio, na escala da resposta que a rede vê, mais a penalidade, calculada lote a lote enquanto os pesos mudam.
> - **`teste_y.median()`** — a mediana.

O ajuste emitiu um `ConvergenceWarning`: o método chegou ao limite de iterações, 200 épocas no padrão, sem convergir. Com lotes de até 200 jogadores, o padrão, os 175 de treino formam um lote só: cada época é um único passo de gradiente, e as 200 épocas são 200 passos.

A perda da primeira época, em milhares de dólares ao quadrado, é praticamente a de prever zero para todo jogador, metade da média dos salários ao quadrado; a razão entre as duas é 1,00. Os pesos iniciais são sorteados perto de zero, e a rede começa prevendo perto de zero. Com a taxa de aprendizado padrão, a perda da 200ª época ainda é 0,93 da primeira, e a maior previsão de teste é de 121 mil dólares, quando metade dos jogadores de teste ganha mais de 387,5. O MAE de teste, 471,09, é pior que o da média.

Há duas saídas: aumentar o limite de épocas ou pôr a resposta em outra escala. A primeira, com o salário ainda em milhares de dólares e um limite dez vezes maior:

In [ ]:
mais_epocas = clone(so_X).set_params(
    rede__max_iter=2000)
avisos = ajusta_e_avisa(mais_epocas, treino_X, treino_y)
print("avisos:", len(avisos), *avisos)
print("épocas:", mais_epocas[-1].n_iter_)
prev_mais = mais_epocas.predict(teste_X)
mae_mais = mean_absolute_error(teste_y, prev_mais)
print("MAE:", numero_br(mae_mais))
print("pior que a linear:", mae_mais > mae_linear)

> **🔧 Função**
>
> - **`clone(modelo)`** — uma cópia do modelo com os mesmos parâmetros, ainda sem ajuste.
> - **`.set_params(nome=valor)`** — muda parâmetros; num `Pipeline`, o nome diz o caminho, com os nomes separados por dois sublinhados: `rede__max_iter` é o `max_iter` do passo `rede`.

Com 2.000 passos, o aviso continua: a rede usou as 2.000 épocas sem convergir. O MAE de teste caiu para 238,90, mas ficou pior que o da regressão linear.

A outra saída é mudar a escala da resposta. O `TransformedTargetRegressor` padroniza a resposta antes do ajuste, e a rede aprende a prever o salário em desvios padrão a partir da média; na previsão, ele desfaz a padronização e devolve milhares de dólares. Para ver o que a padronização dos preditores faz, o primeiro ajuste com ele padroniza **só** a resposta e deixa as 19 colunas como estão. E se os preditores ficarem crus, a rede erra quanto?

In [ ]:
so_y = TransformedTargetRegressor(
    regressor=MLPRegressor(
        hidden_layer_sizes=(64,), random_state=14),
    transformer=StandardScaler())
avisos = ajusta_e_avisa(so_y, treino_X, treino_y)
print("avisos:", len(avisos), *avisos)
print("épocas:", so_y.regressor_.n_iter_)
z = StandardScaler().fit_transform(
    treino_y.to_frame())
print("perda de prever a média:",
    numero_br(0.5 * (z ** 2).mean()))
perda_y = np.array(
    so_y.regressor_.loss_curve_)
menor = perda_y.argmin()
print("menor perda na época", menor + 1)
print("  perda:", numero_br(perda_y[menor], 0))
depois = perda_y[menor + 1:]
print("  depois, todas acima:",
    (depois > perda_y[menor]).all())
prev_so_y = so_y.predict(teste_X)
print("MAE:", numero_br(
    mean_absolute_error(teste_y, prev_so_y), 0))

> **🔧 Função**
>
> - **`TransformedTargetRegressor(regressor, transformer)`** — ajusta o `regressor` na resposta transformada pelo `transformer`, aqui a padronização, e desfaz a transformação nas previsões. **`.regressor_`** — o regressor já ajustado, lá dentro.
> - **`.fit_transform(x)`** — ajusta o padronizador em `x` e devolve `x` já padronizado.
> - **`serie.to_frame()`** — a série como uma tabela de uma coluna, a forma que o `StandardScaler` recebe.
> - **`v.argmin()`** — a posição do menor valor de `v`, contada a partir de 0; por isso a época é `menor + 1`.

O ajuste parou em 29 épocas, sem aviso. Na escala padronizada da resposta, prever a média para todo jogador custa uma perda de 0,50; a menor perda da rede foi de 2.702, na época 18, e todas as seguintes ficaram acima dela. Sem uma melhora de ao menos uma pequena tolerância por mais de 10 épocas seguidas, o `MLPRegressor` para, e parar sem aviso, aqui, não quer dizer que o ajuste convergiu: o MAE de teste é de 56.829 mil dólares. Os pesos iniciais de todas as colunas são sorteados no mesmo intervalo, e, na soma $w_{k0} + \sum_j w_{kj}X_j$ de cada unidade, a coluna que chega a 14.053 pesa milhares de vezes mais que a indicadora de 0 ou 1.

Com as duas escalas, preditores e resposta padronizados, a rede de 64 unidades ajusta assim, com o limite padrão de 200 épocas e com o de 2.000. A função `rede_padronizada` monta o pipeline e repassa ao `MLPRegressor` os parâmetros dados por nome:

In [ ]:
def rede_padronizada(**parametros):
    rede = MLPRegressor(
        random_state=14, **parametros)
    return Pipeline([
        ("escala", StandardScaler()),
        ("rede", TransformedTargetRegressor(
            rede, transformer=StandardScaler())),
    ])

redes = {}
for limite in (200, 2000):
    redes[limite] = rede_padronizada(
        hidden_layer_sizes=(64,), max_iter=limite)
    avisos = ajusta_e_avisa(
        redes[limite], treino_X, treino_y)
    ajustada = redes[limite][-1].regressor_
    print(f"max_iter {limite}:")
    print("  avisos:", len(avisos), *avisos)
    print("  épocas:", ajustada.n_iter_)
prev_escala = redes[2000].predict(teste_X)
mae_escala = mean_absolute_error(teste_y, prev_escala)
print("MAE com 2.000:", numero_br(mae_escala))
print("abaixo da linear:", mae_escala < mae_linear)
print("abaixo da de mil US$:", mae_escala < mae_mais)

Com o limite padrão de 200 épocas, o `ConvergenceWarning` volta: o ajuste ainda não tinha convergido. Com o limite de 2.000, ele para sozinho em 491 épocas, sem aviso, e o MAE de teste fica em 213,77, abaixo do da regressão linear e do da rede com o salário em milhares de dólares e o mesmo limite. O mesmo aviso apareceu nas duas redes de 200 épocas, com causas diferentes: com o salário em milhares de dólares, nem 2.000 épocas bastaram; com as duas escalas, bastou aumentar o limite.

A figura compara as previsões de teste dessas duas redes de 64 unidades com limite de 2.000, que diferem só na escala da resposta:

In [ ]:
# Figura: Salário previsto contra o observado dos 88 jogadores de teste, em milhares de dólares, para a rede de 64 unidades com os preditores padronizados e o limite de 2.000 épocas: com a resposta em milhares de dólares (em cima) e com a resposta padronizada (embaixo). Na linha cinza tracejada, previsão igual ao observado.
eixo = (-500, 2500)
paineis = [
    (prev_mais, "resposta em mil US$"),
    (prev_escala, "resposta padronizada"),
]
for previsao, titulo in paineis:
    print(titulo + ":")
    print("  MAE:", numero_br(
        mean_absolute_error(teste_y, previsao)))
    print("  abaixo do observado:",
        (previsao < teste_y).sum(), "de", len(teste_y))
    print("  previsões negativas:", (previsao < 0).sum())
    print("  a menor:", numero_br(previsao.min(), 0))
dentro = [
    eixo[0] < p.min() and p.max() < eixo[1]
    for p in (teste_y, prev_mais, prev_escala)]
print("tudo dentro do eixo:", all(dentro))
fig, eixos = plt.subplots(
    2, 1, figsize=(3.8, 6.0),
    sharex=True, sharey=True)
for ax, (previsao, titulo) in zip(eixos, paineis):
    ax.plot(eixo, eixo, color="0.45",
        linestyle="--", linewidth=1)
    ax.scatter(teste_y, previsao, s=14, color="C1",
        alpha=0.7, linewidths=0)
    ax.set_title(titulo)
    ax.set_ylabel("previsto (mil US$)")
    ax.set_xlim(*eixo)
    ax.set_ylim(*eixo)
    ax.set_aspect("equal")
eixos[1].set_xlabel("observado (mil US$)")
for ax in eixos:
    for lado in (ax.xaxis, ax.yaxis):
        lado.set_major_formatter(
            lambda v, _: numero_br(v, 0))
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`sharey=True`** — no `plt.subplots`, os gráficos dividem também o eixo vertical, e as nuvens de pontos se comparam na mesma escala.
> - **`ax.set_aspect("equal")`** — uma unidade vale o mesmo comprimento nos dois eixos, e a diagonal fica a 45 graus.

As duas nuvens se parecem, e a figura sozinha não diz qual rede é melhor; o MAE de teste, 238,90 contra 213,77, diz. Nenhuma das duas impede uma previsão abaixo de zero: com a resposta em milhares de dólares, 1 jogador de teste tem previsão de salário negativa; com a resposta padronizada, 5, a menor de −212 mil dólares. Os dois painéis mostram todos os jogadores de teste: o maior salário e todas as previsões cabem entre −500 e 2.500.

### Escolher o tamanho e a penalidade

A rede de 64 unidades com `alpha=0.0001`, o padrão, foi um palpite: nem o número de unidades nem a força da penalidade foram escolhidos. A penalidade é a da ridge sobre os pesos, da seção 14.3, e o $\lambda$ que ela representa depende do tamanho do lote. O lote padrão tem $b = \min(200, n)$ jogadores. Com 175 no treino, ou 140 no ajuste de cada rodada de uma validação cruzada de cinco grupos, ele é o treino inteiro:

In [ ]:
for n in (len(treino_X), len(treino_X) * 4 // 5):
    print(f"n = {n}: lote de {min(200, n)}")

Com um lote só, cada época é um passo, e a perda desse passo no `MLPRegressor` é metade do erro quadrático médio mais $\texttt{alpha}/(2n)$ vezes a soma dos quadrados dos pesos. Multiplicada por $2n$, ela vira
$$
\mathrm{RSS} + \lambda \sum_j \theta_j^2, \qquad \lambda = \texttt{alpha},
$$
a soma dos quadrados dos resíduos, na escala padronizada do salário, mais $\lambda$ vezes a soma dos quadrados dos pesos, sem os vieses. Nessa forma, com o RSS e sem o ½ do `scikit-learn`, o número passado como `alpha` é o próprio $\lambda$. Na seção 14.3, com lotes menores que o treino, a conversão de `alpha` em $\lambda$ dependia da razão entre o treino e o lote; aqui, `alpha=1` vale $\lambda = 1$. O mesmo número em `alpha` é uma penalidade diferente em cada problema, mais um motivo para escolhê-lo pelos dados.

Duas escolhas, portanto: o número de unidades da camada escondida e $\lambda$. A validação cruzada de cinco grupos da seção 10.3 escolhe as duas juntas, sem tocar nos 88 jogadores de teste. A grade tem sete valores de `alpha`, de 0,1 (mil vezes o padrão) a 100, igualmente espaçados em escala log, e dois tamanhos de camada, 16 e 64 unidades; o critério é o MAE. Se a grade for larga o bastante, o erro de validação cruzada sobe nas duas pontas. A camada de 16 unidades erra mais ou menos que a de 64?

In [ ]:
caminho = "rede__regressor__"
grade = {
    caminho + "alpha": np.logspace(-1, 2, 7),
    caminho + "hidden_layer_sizes": [(16,), (64,)],
}
metrica = "neg_mean_absolute_error"
busca = GridSearchCV(
    rede_padronizada(
        hidden_layer_sizes=(16,), max_iter=2000),
    grade, cv=KFold(5, shuffle=True, random_state=14),
    scoring=metrica)
busca.fit(treino_X, treino_y)
print("candidatos:", len(busca.cv_results_["params"]))
escolha = busca.best_params_
print("camada:", escolha[caminho + "hidden_layer_sizes"])
print("alpha:", numero_br(escolha[caminho + "alpha"]))
print("MAE de validação cruzada:", numero_br(-busca.best_score_))
prev_busca = busca.predict(teste_X)
mae_busca = mean_absolute_error(teste_y, prev_busca)
print("MAE de teste:", numero_br(mae_busca))
print("R² de teste:", numero_br(r2_score(teste_y, prev_busca)))
final = busca.best_estimator_[-1].regressor_
n_coefs = sum(c.size for c in final.coefs_)
n_vieses = sum(v.size for v in final.intercepts_)
print("pesos:", n_coefs + n_vieses)
print("épocas:", final.n_iter_)

> **🔧 Função**
>
> - **`GridSearchCV(modelo, grade, cv, scoring)`** — ajusta o modelo com cada combinação de valores da `grade` em cada grupo da validação cruzada `cv`, e fica com a combinação de melhor nota média; no fim, reajusta essa combinação com todo o treino.
> - As chaves da grade dizem o caminho até o parâmetro, como no `set_params`: `rede__regressor__alpha` é o `alpha` do `regressor` do passo `rede`, e `caminho` guarda a parte comum.
> - `scoring="neg_mean_absolute_error"`, guardado em `metrica`, usa o MAE com o sinal trocado, porque a busca procura a maior nota.
> - **`KFold(5, shuffle=True, random_state=14)`** — os cinco grupos, com os jogadores embaralhados com semente 14 antes da divisão.
> - **`np.logspace(-1, 2, 7)`** — sete números de $10^{-1}$ a $10^{2}$, igualmente espaçados na escala log.
> - **`busca.best_params_`** — a combinação escolhida; **`busca.best_score_`**, a nota média dela nos grupos; **`busca.cv_results_`**, os resultados de todos os candidatos; **`busca.best_estimator_`**, o modelo reajustado com todo o treino.

A busca ajustou 14 candidatos, cada um em cinco grupos, e escolheu a camada de 16 unidades com `alpha` 3,16, combinação que dá um MAE de validação cruzada de 206,14 mil dólares. Reajustada com os 175 jogadores, essa rede tem 337 pesos, para em 859 épocas e erra, em média, 178,34 mil dólares nos jogadores de teste, com $R^2$ de 0,65. A rede de 64 unidades sem escolha nenhuma tinha errado 213,77, e a linear, 232,06.

O MAE de teste, 178,34, ficou abaixo do de validação cruzada, 206,14. Os dois oscilam: o de teste vem de 88 jogadores numa divisão só, e o de validação cruzada, da média de cinco grupos de 35. Por isso as comparações acima são entre MAE de teste, nos mesmos 88 jogadores.

A figura mostra o MAE de validação cruzada de cada candidato contra `alpha`, uma curva por tamanho de camada. O que olhar é a forma de cada curva: se ela desce até um fundo e volta a subir, a grade alcançou as duas pontas.

In [ ]:
# Figura: MAE de validação cruzada (cinco grupos, em milhares de dólares) contra `alpha`, em escala log, para a camada escondida de 16 unidades (azul) e de 64 unidades (laranja). O x marca o candidato escolhido.
tira = lambda c: c.replace("param_" + caminho, "")
resultados = pd.DataFrame(busca.cv_results_)
resultados = resultados.rename(columns=tira)
tamanhos = resultados["hidden_layer_sizes"]
tabela = resultados.pivot_table(
    index="alpha",
    columns=tamanhos.map(lambda t: f"{t[0]} un."),
    values="mean_test_score") * -1
fig, ax = plt.subplots(figsize=(3.8, 3.2))
for coluna, cor in [("16 un.", "C0"), ("64 un.", "C1")]:
    ax.plot(tabela.index, tabela[coluna], color=cor,
        marker="o", markersize=4, label=coluna)
ax.plot(escolha[caminho + "alpha"], -busca.best_score_,
    "x", color="C0", markersize=11, markeredgewidth=2.5,
    zorder=3)
ax.set_xscale("log")
ax.set_xlim(0.06, 170)
ax.set_xlabel("alpha")
ax.set_ylabel("MAE de validação cruzada")
ax.xaxis.set_major_formatter(
    lambda v, _: f"{v:g}".replace(".", ","))
ax.yaxis.set_major_formatter(lambda v, _: numero_br(v, 0))
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`df.rename(columns=função)`** — troca o nome de cada coluna pelo que a função devolve; aqui, a função `tira` remove o prefixo `param_rede__regressor__`.
> - **`df.pivot_table(index, columns, values)`** — rearranja os resultados numa tabela com um valor de `index` por linha e um de `columns` por coluna, com `values` nas células; **`* -1`** desfaz o sinal trocado do `neg_mean_absolute_error`.
> - **`.map(lambda t: f"{t[0]} un.")`** — escreve cada tupla de tamanhos como texto, `(16,)` virando "16 un.".
> - **`ax.set_xscale("log")`** — o eixo horizontal em escala log: a distância entre 0,1 e 1 é a mesma que entre 1 e 10.
> - **`ax.plot(..., marker="o", markersize=4)`** — marca cada valor da grade com um ponto na linha; **`markeredgewidth=2.5`** é a espessura do traço do marcador, aqui o x.

O que a figura mostra, conferido nos números:

In [ ]:
curva_16 = tabela["16 un."]
abaixo = curva_16 < tabela["64 un."]
print("16 un. abaixo de 64 un.:",
    abaixo.sum(), "de", len(tabela))
alphas_abaixo = tabela.index[abaixo.to_numpy()]
print("  nos alpha:",
    ", ".join(numero_br(a) for a in alphas_abaixo))
fundo = curva_16.to_numpy().argmin()
print("16 un., fundo em", numero_br(tabela.index[fundo]))
desce = (np.diff(curva_16.iloc[:fundo + 1]) < 0).all()
sobe = (np.diff(curva_16.iloc[fundo:]) > 0).all()
print("  cai até ele e sobe depois:", bool(desce and sobe))
print("  em 0,10:", numero_br(curva_16.iloc[0]))
print("  em 100,00:", numero_br(curva_16.iloc[-1]))
grandes = tabela.loc[tabela.index >= 10]
distancia = (grandes["16 un."] - grandes["64 un."]).abs()
print("maior distância em alpha ≥ 10:",
    numero_br(distancia.max()))

> **🔧 Função**
>
> - **`np.diff(v)`** — a diferença de cada valor para o anterior.
> - **`serie.abs()`** — o valor absoluto de cada elemento.

A camada de 16 unidades fica abaixo da de 64 em 4 dos 7 valores de `alpha`, os quatro menores, de 0,10 a 3,16. A curva de 16 unidades cai até o fundo, em 3,16, e sobe depois dele, até 277,98 em 100; na outra ponta, em 0,10, fica em 214,55. Com penalidade de menos ou de mais, o erro sobe, como na ridge da seção 11.3, e a grade alcança as duas pontas. A vantagem das 16 unidades vale para esta grade, nestes jogadores; com outra divisão em grupos, as curvas podem mudar.

### Parada antecipada

A parada antecipada da seção 14.3 existe como opção do `MLPRegressor` e do `MLPClassifier`. Com `early_stopping=True`, a rede separa uma parte do treino para validação, mede o $R^2$ nela depois de cada época e para quando ele deixa de melhorar. Na rede escolhida pela busca, a opção fica ligada com um quinto do treino para validação e 20 épocas de paciência. Com 35 jogadores a menos no ajuste dos pesos, a rede erra mais ou menos no teste?

In [ ]:
com_parada = clone(busca.best_estimator_)
com_parada[-1].regressor.set_params(
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=20)
com_parada.fit(treino_X, treino_y)
parou = com_parada[-1].regressor_
print("jogadores de validação:",
    math.ceil(0.2 * len(treino_X)))
print("épocas:", parou.n_iter_)
r2 = pd.Series(parou.validation_scores_,
    index=range(1, parou.n_iter_ + 1))
print("melhor R² de validação:", numero_br(r2.max(), 4))
print("  na época", r2.idxmax())
print("  é a última:", r2.idxmax() == parou.n_iter_)
print("tol:", numero_br(parou.tol, 4))
melhor_antes = r2.cummax().shift()
sem_melhora = r2 < melhor_antes + parou.tol
melhoras = sem_melhora[~sem_melhora]
ultima = melhoras.index.max()
print("última melhora ≥ tol:", ultima)
print("nas 21 últimas épocas,")
print("  nenhuma melhora ≥ tol:",
    sem_melhora.iloc[-21:].all())
previsto_p = com_parada.predict(teste_X)
mae_parada = mean_absolute_error(teste_y, previsto_p)
print("MAE de teste:", numero_br(mae_parada))
print("acima do sem parada:", mae_parada > mae_busca)

> **🔧 Função**
>
> - **`com_parada[-1].regressor`** — o `MLPRegressor` dentro do `TransformedTargetRegressor`, ainda sem ajuste; **`.set_params`** muda os parâmetros dele ali mesmo. O `.regressor_`, com o sublinhado, é a cópia ajustada.
> - No `MLPRegressor`: **`early_stopping=True`** liga a parada antecipada; **`validation_fraction`** é a fração do treino separada para validação (o padrão é 0,1); **`n_iter_no_change`** é a paciência, em épocas (o padrão é 10).
> - **`.validation_scores_`** — o $R^2$ de validação depois de cada época.
> - **`serie.cummax()`** — em cada posição, o maior valor até ali; **`.shift()`** empurra a série uma posição para baixo, e cada época fica com o melhor valor das anteriores.
> - **`sem_melhora[~sem_melhora]`** — as épocas em que `sem_melhora` é `False`, as de melhora de ao menos `tol`; `~` troca `True` por `False` e vice-versa.
> - **`math.ceil(x)`** — arredonda `x` para cima, como o `scikit-learn` faz com o tamanho da validação.

Na regra de parada, uma época conta como melhora só se superar o melhor valor até ali por ao menos uma tolerância, `tol`; o ajuste para quando passam mais de `n_iter_no_change` épocas seguidas sem melhora, e os pesos guardados são os da época de maior valor de validação. No classificador, o valor medido é a taxa de acerto, e não a perda, e a seção 14.3 mostrou que esses dois critérios podem levar a paradas diferentes.

São 35 jogadores de validação, e o ajuste parou em 526 épocas. A última melhora de ao menos `tol`, 0,0001, foi na época 505; nas 21 seguintes, uma a mais que a paciência de 20, nenhuma época superou o melhor $R^2$ das anteriores por ao menos isso, e o ajuste parou. Mesmo assim, a melhor época foi a própria última, com $R^2$ de validação de 0,7200: ela superou as anteriores, mas por menos que `tol`. Uma melhora pequena assim não zera a contagem da paciência, mas os pesos guardados são sempre os do maior $R^2$; como ele foi o da última época, não houve o que desfazer.

In [ ]:
# Figura: R² de validação (35 jogadores) depois de cada época, na rede de 16 unidades com `alpha` 3,16 e parada antecipada. A faixa cinza cobre as 21 épocas depois da última melhora de ao menos `tol`, a 505; o x marca a época de maior R², que é também a época em que o ajuste parou.
print("R² na época 1:", numero_br(r2.iloc[0], 2))
print("  na época 100:", numero_br(r2.loc[100], 2))
print("  na época 426:", numero_br(r2.loc[426], 4))
print("1ª época acima de 0:", (r2 > 0).idxmax())
fig, ax = plt.subplots(figsize=(3.8, 3.0))
ax.axvspan(ultima, r2.index[-1], color="0.45",
    alpha=0.25, linewidth=0)
ax.plot(r2.index, r2, color="C0", linewidth=1.5)
ax.plot(r2.idxmax(), r2.max(), "x", color="C1",
    markersize=10, markeredgewidth=2.5, zorder=3)
traco = {"arrowstyle": "-", "color": "0.45"}
ax.annotate(f"melhor e última: {r2.idxmax()}",
    (r2.idxmax(), r2.max()), xytext=(500, -0.9),
    ha="right", va="center", arrowprops=traco)
ax.axhline(0, color="0.45", linewidth=0.8, zorder=0)
ax.set_xlim(0, 545)
ax.set_xlabel("época")
ax.set_ylabel("R² de validação")
ax.yaxis.set_major_formatter(lambda v, _: numero_br(v, 1))
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`ax.axvspan(x0, x1)`** — pinta uma faixa vertical entre `x0` e `x1`; `alpha` é a opacidade.
> - **`"arrowstyle": "-"`** — no `arrowprops` do `ax.annotate`, troca a seta por um traço sem ponta, do texto em `xytext` até o ponto marcado.

O $R^2$ de validação começa em −1,71 na época 1, pior que prever a média, que daria perto de 0, e passa de 0 na época 61. Na época 100 já está em 0,34; nas últimas 100 épocas, da 426 à 526, ganha só 0,0092 (de 0,7108 a 0,7200). A curva quase parou de subir, e é nesse trecho que a paciência se esgota.

O MAE de teste ficou em 193,42, contra 178,34 da mesma rede sem parada. A parada antecipada tira 35 dos 175 jogadores do ajuste dos pesos em troca de um critério de parada, e nesta divisão a troca não compensou.

### O mesmo para classificar

O `MLPClassifier` segue o mesmo caminho. A tabela `Heart` tem 13 medidas de pacientes com dor no peito e, em `doenca_cardiaca`, o resultado de uma angiografia: se ela confirmou ou não doença cardíaca. A busca é do mesmo tipo, com nove valores de `alpha`, de 0,01 a 100, e os mesmos dois tamanhos de camada. Os pacientes ficam divididos ao meio entre treino e teste, na mesma proporção de doentes, e o critério é a taxa de erro de validação cruzada:

In [ ]:
coracao = pd.read_csv(
    "dados/Heart.csv").dropna()
alvo = "doenca_cardiaca"
y_c = coracao[alvo]
X_c = pd.get_dummies(
    coracao.drop(columns=alvo), drop_first=True,
    dtype=float)
treino_Xc, teste_Xc, treino_yc, teste_yc = train_test_split(
    X_c, y_c, test_size=0.5, random_state=14, stratify=y_c)
print("pacientes:", len(X_c), "· medidas:", coracao.shape[1] - 1)
print("treino:", len(treino_Xc), "· teste:", len(teste_Xc))
classificador = Pipeline([
    ("escala", StandardScaler()),
    ("rede", MLPClassifier(
        hidden_layer_sizes=(16,), max_iter=2000,
        random_state=14)),
])
grupos = StratifiedKFold(5, shuffle=True, random_state=14)
camada_c = "rede__hidden_layer_sizes"
grade_c = {
    "rede__alpha": np.logspace(-2, 2, 9),
    camada_c: [(16,), (64,)],
}
busca_c = GridSearchCV(
    classificador, grade_c, cv=grupos, scoring="accuracy")
busca_c.fit(treino_Xc, treino_yc)
print("candidatos:", len(busca_c.cv_results_["params"]))
escolha_c = busca_c.best_params_
print("camada:", escolha_c[camada_c])
print("alpha:", numero_br(escolha_c["rede__alpha"]))
print("erro de validação cruzada:",
    numero_br(1 - busca_c.best_score_, 4))
erros_rede = (busca_c.predict(teste_Xc) != teste_yc).sum()
print("erros de teste:", erros_rede, "de", len(teste_yc))

> **🔧 Função**
>
> - **`StratifiedKFold(5, shuffle=True, random_state=14)`** — os cinco grupos, com a mesma proporção de doentes em cada um.
> - **`scoring="accuracy"`** — a nota é a taxa de acerto, e 1 menos ela é a taxa de erro.

Dos 297 pacientes, 148 ficam no treino e 149 no teste. Com lotes de até 200, o lote volta a ser o treino inteiro. O `MLPClassifier` minimiza a entropia cruzada **média** mais $\texttt{alpha}/(2n)$ vezes a soma dos quadrados dos pesos. Essa perda não tem o ½ da regressão, e por isso aqui $\lambda$ vale a metade de `alpha`: multiplicada por $n$, ela vira a entropia cruzada somada da seção 14.3 mais $\lambda = \texttt{alpha}/2$ vezes a soma dos quadrados dos pesos. É a conta $\lambda = \texttt{alpha}\cdot n/(2b)$ daquela seção, com $b = n$.

A busca comparou 18 candidatos e escolheu a camada de 16 unidades com `alpha` 31,62, com erro de validação cruzada de 0,1821; a rede reajustada erra 27 dos 149 pacientes de teste. A tabela da busca vem abaixo e, depois dela, a taxa de erro de validação cruzada de uma referência que prevê "não" para todos os pacientes:

In [ ]:
res_c = pd.DataFrame(busca_c.cv_results_)
res_c = res_c.rename(
    columns=lambda c: c.replace("param_rede__", ""))
tamanhos_c = res_c["hidden_layer_sizes"]
erro_c = 1 - res_c.pivot_table(
    index="alpha",
    columns=tamanhos_c.map(lambda t: f"{t[0]} un."),
    values="mean_test_score")
erro_c.index = [numero_br(a) for a in erro_c.index]
erro_c.index.name = "alpha"
erro_c.columns.name = None
print(erro_c.map(lambda v: numero_br(v, 4)).to_string())
empate = (erro_c.loc["31,62", "16 un."]
    == erro_c.loc["31,62", "64 un."])
print("empate em 31,62:", empate)
sempre_nao = DummyClassifier(
    strategy="constant", constant="não")
acertos_nao = cross_val_score(
    sempre_nao, treino_Xc, treino_yc,
    cv=grupos, scoring="accuracy")
erro_nao = 1 - acertos_nao.mean()
print("prever sempre 'não':", numero_br(erro_nao, 4))
erro_100 = erro_c.loc["100,00", "16 un."]
print("igual a alpha 100 (4 casas):",
    round(erro_nao, 4) == round(erro_100, 4))

> **🔧 Função**
>
> - **`DummyClassifier(strategy="constant", constant="não")`** — um classificador que ignora as colunas e prevê sempre a classe `constant`; serve de referência.
> - **`cross_val_score(modelo, X, y, cv, scoring)`** — a nota do modelo em cada grupo da validação cruzada; `.mean()` tira a média dos cinco grupos.

Na grade, o erro não cai de forma regular com `alpha`: na camada de 16 unidades, ele sobe de 0,2028 para 0,2368 em `alpha` 1, desce até 0,1821 em 31,62 e salta para 0,4593 em 100. Em 31,62, as duas camadas empatam, e a busca fica com o que vem antes na ordem da grade, o de 16 unidades. Em `alpha` 100, o erro de 0,4593 é, com quatro casas, o de prever "não" para todos os pacientes: com a penalidade forte demais, a rede não faz melhor que ignorar as medidas.

A referência linear é a regressão logística sem penalidade, avaliada nos mesmos grupos:

In [ ]:
logistica = Pipeline([
    ("escala", StandardScaler()),
    ("logistica", LogisticRegression(
        C=np.inf, solver="newton-cholesky",
        tol=1e-8)),
])
acertos = cross_val_score(
    logistica, treino_Xc, treino_yc,
    cv=grupos, scoring="accuracy")
print("logística, validação cruzada:",
    numero_br(1 - acertos.mean(), 4))
logistica.fit(treino_Xc, treino_yc)
erros_log = (logistica.predict(teste_Xc) != teste_yc).sum()
print("logística, erros de teste:", erros_log)
print("diferença no teste:", abs(erros_rede - erros_log), "paciente(s)")

> **🔧 Função**
>
> **`LogisticRegression(C=np.inf, solver="newton-cholesky", tol=1e-8)`** — a logística sem penalidade (`C` infinito), com um método de busca e uma tolerância que levam ao mesmo ponto em máquinas diferentes.

A logística sem penalidade tem erro de validação cruzada de 0,2297 e erra 26 pacientes de teste. Os dois erros de validação cruzada não se comparam de igual para igual: o 0,1821 da rede é o menor de 18 candidatos medidos nos mesmos grupos e, como na seção 11.5, tende a sair otimista; o da logística não passou por escolha nenhuma. No teste, que nenhuma das duas usou para escolher, a rede escolhida e a logística ficam a um paciente de distância, com 148 pacientes para ajustar e 149 para medir: nestes dados, a rede não mostra vantagem sobre o modelo linear.

E a parada antecipada, com os valores padrão, na rede escolhida pela busca? Com 148 pacientes de treino, ela ajuda ou atrapalha?

In [ ]:
parada_c = clone(busca_c.best_estimator_)
parada_c.set_params(
    rede__early_stopping=True)
parada_c.fit(treino_Xc, treino_yc)
mlp = parada_c[-1]
print("camada:", mlp.hidden_layer_sizes,
    "· alpha:", numero_br(mlp.alpha))
fracao = mlp.validation_fraction
print("fração de validação:", numero_br(fracao, 1))
print("paciência:", mlp.n_iter_no_change, "épocas")
n_val = math.ceil(fracao * len(treino_Xc))
print("pacientes de validação:", n_val)
print("épocas:", mlp.n_iter_)
acerto_v = pd.Series(mlp.validation_scores_,
    index=range(1, mlp.n_iter_ + 1))
print("acerto na época 1:", numero_br(acerto_v.iloc[0], 3))
print("  alguma época acima dela:",
    (acerto_v > acerto_v.iloc[0]).any())
print("pesos guardados: época", acerto_v.idxmax())
previsto_c = parada_c.predict(teste_Xc)
print("erros de teste:", (previsto_c != teste_yc).sum())
print("prevendo sempre 'não':", (teste_yc != "não").sum())
print("previsões de teste:")
contagem = pd.Series(previsto_c).value_counts()
print(contagem.to_string())

A rede é a escolhida pela busca, com 16 unidades e `alpha` 31,62, agora com a parada antecipada ligada. Com a fração padrão, 0,1, a validação fica com 15 dos 148 pacientes de treino. O acerto nela, na primeira época, é de 0,467, 7 dos 15, e nenhuma das 11 épocas seguintes, até a 12ª, passa dele: a primeira define o melhor valor, e a paciência se esgota antes de outra superá-lo. Os pesos voltam aos de depois da primeira época. A rede prevê "sim" para 20 pacientes de teste e erra 69 dos 149, o mesmo número de erros de prever "não" para todos.

Com o lote igual ao treino, cada época é um único passo de gradiente, e a paciência, contada em épocas, conta passos. A parada veio de poucos passos, ou de uma validação pequena demais? A primeira contraprova mantém os mesmos 15 pacientes de validação e aumenta a paciência para 100 épocas:

In [ ]:
longa = clone(parada_c).set_params(
    rede__n_iter_no_change=100)
longa.fit(treino_Xc, treino_yc)
mlp_longa = longa[-1]
fracao_l = mlp_longa.validation_fraction
print("pacientes de validação:",
    math.ceil(fracao_l * len(treino_Xc)))
acerto_l = pd.Series(
    mlp_longa.validation_scores_,
    index=range(1, mlp_longa.n_iter_ + 1))
antes = mlp.n_iter_
print(f"iguais até a época {antes}:")
inicio_l = acerto_l.iloc[:antes]
print("  acerto de validação:",
    inicio_l.equals(acerto_v))
perda_l = mlp_longa.loss_curve_[:antes]
print("  perda de treino:", perda_l == mlp.loss_curve_)
print("acerto na época 1:", numero_br(acerto_l.iloc[0], 3))
primeira = (acerto_l > acerto_l.iloc[0]).idxmax()
print("  superado pela 1ª vez na época", primeira)
print("épocas:", mlp_longa.n_iter_)
print("melhor época:", acerto_l.idxmax())
print("  acerto:", numero_br(acerto_l.max(), 3))
print("  acertos:", round(acerto_l.max() * n_val),
    "de", n_val)
erros_longa = (longa.predict(teste_Xc) != teste_yc).sum()
print("erros de teste:", erros_longa)
print("acerto de teste:",
    numero_br(1 - erros_longa / len(teste_yc), 3))

> **🔧 Função**
>
> **`serie.equals(outra)`** — `True` se as duas séries têm os mesmos rótulos e os mesmos valores, posição a posição.

Até a época 12, as duas redes são a mesma: a mesma perda de treino e o mesmo acerto de validação em cada época. A diferença é só a paciência. Com 100 épocas dela, o acerto de validação só supera o da primeira época na época 82, bem depois da 12ª, em que a paciência de 10 tinha parado o ajuste. A rede sai do lugar e erra 29 dos 149 pacientes de teste, contra 69 com a paciência padrão.

Com 15 pacientes, o acerto de validação anda de 1/15 em 1/15, e acertar 13 dos 15 na melhor época não quer dizer acertar os pacientes novos na mesma proporção: no teste, a rede erra 29 de 149, um acerto de 0,805.

A segunda contraprova faz o contrário: mantém a paciência padrão, de 10 épocas, e aumenta a validação para 0,2 e 0,3 do treino:

In [ ]:
for fracao in (0.2, 0.3):
    maior = clone(parada_c)
    maior[-1].set_params(
        validation_fraction=fracao)
    maior.fit(treino_Xc, treino_yc)
    rede_m = maior[-1]
    acerto_m = pd.Series(rede_m.validation_scores_)
    n_maior = math.ceil(fracao * len(treino_Xc))
    erros_m = (maior.predict(teste_Xc) != teste_yc).sum()
    print(f"validação de {n_maior} pacientes:")
    print("  épocas:", rede_m.n_iter_)
    print("  melhor época:", acerto_m.idxmax() + 1)
    print("  erros de teste:", erros_m)

Com 30 pacientes de validação, a rede tem a melhor época na 7ª, para na 18ª e erra 68 pacientes de teste; com 45, a melhor época é a primeira, e ela para na 12ª e erra 69. Nas duas, o erro de teste fica a no máximo um paciente dos 69 da paciência padrão, longe dos 29 da paciência de 100. Nesta divisão, a validação maior não tirou a rede do lugar, e a paciência maior tirou: o que a parou na 12ª época foi uma paciência de 10 épocas que aqui são 10 passos.

### A receita

Os passos dos dois exemplos:

- **Padronizar os preditores** num `Pipeline`: sem isso, a rede de regressão errou 56.829 mil dólares em média.
- **Padronizar a resposta**, na regressão, com o `TransformedTargetRegressor`: com o salário em milhares de dólares, nem 2.000 épocas bastaram, e o MAE de teste, 238,90, ficou acima do da regressão linear.
- **Ler o `ConvergenceWarning`** antes de aumentar `max_iter`: o mesmo aviso apareceu com a resposta fora de escala, que 2.000 épocas não resolveram, e, com as escalas certas, por falta de épocas.
- **Escolher o tamanho da camada e `alpha` juntos, por validação cruzada**, com o teste de fora.
- **Comparar com uma referência simples**: no `Hitters`, a rede de 16 unidades erra 178,34 mil dólares, contra 232,06 da linear e 344,05 da média; no `Heart`, fica a um paciente da logística.
- **Fixar a semente** (`random_state`), para que o ajuste, que começa de pesos sorteados, possa ser repetido.
- **Na parada antecipada, contar os passos da paciência**: com o lote igual ao treino, cada época é um passo, e a paciência padrão de 10 épocas parou a rede do `Heart` na 12ª, antes de o acerto de validação superar o da primeira época; com 100 épocas de paciência, ela errou 29 pacientes de teste em vez de 69, e aumentar só a validação não bastou.
- **Descontar a validação do treino**: a parte separada para validação sai do ajuste dos pesos, e na regressão a rede com parada, ajustada com 35 jogadores a menos, errou mais no teste que a mesma rede sem parada.

## Quando Usar Deep Learning

> **📌 Nota**
>
> Esta seção corresponde à seção 10.6 de James et al. (2023).

Nas imagens de dígitos da seção 14.2, uma rede com duas camadas escondidas errou bem menos que a regressão logística. Deve-se então pedir uma rede para todo problema com dados? Nos jogadores de beisebol, com poucas centenas de linhas e 19 colunas, a rede pede padronização dos preditores e da resposta, um limite de épocas, uma busca por validação cruzada e uma semente. Esse trabalho compensa? Contra quais modelos a rede deve ser comparada para saber?

O *profundo* de *deep learning*, aprendizado profundo, se refere ao número de camadas: uma rede com muitas camadas escondidas é profunda, seja ela densa, como as deste capítulo, seja de outro tipo. Com uma ou duas camadas escondidas, as redes deste capítulo têm poucas camadas, mas a pergunta sobre quando usá-las é a mesma que se faz para as profundas: contra modelos mais simples, a rede ganha?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (RandomForestClassifier,
                              RandomForestRegressor)
from sklearn.linear_model import (Lasso, LinearRegression,
                                  LogisticRegression)
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import (GridSearchCV, KFold,
                                     train_test_split)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}"
    .replace(",", "_").replace(".", ",")
    .replace("_", ".").replace("-", "−")
)

### Quatro modelos numa divisão

A tabela `Hitters` traz as estatísticas de 1986 dos jogadores das grandes ligas e o salário de 1987, em milhares de dólares. Descartados os jogadores sem salário e trocadas as colunas de texto por indicadoras, um terço dos jogadores fica para teste:

In [ ]:
hitters = pd.read_csv(
    "dados/Hitters.csv").dropna()
y = hitters["salario"]
X = pd.get_dummies(
    hitters.drop(columns="salario"),
    drop_first=True, dtype=float)
print("jogadores:", len(X))
print("colunas:", X.shape[1])
treino_X, teste_X, treino_y, teste_y = (
    train_test_split(
        X, y, test_size=1/3,
        random_state=14))
print("treino:", len(treino_X))
print("teste:", len(teste_X))

São 175 jogadores de treino e 88 de teste. Quatro modelos disputam a previsão do salário:

- a **regressão linear** por mínimos quadrados, com as 19 colunas;
- o **lasso** da seção 11.4, com as colunas padronizadas e o `alpha` escolhido por validação cruzada de cinco grupos, numa grade de 61 valores de 0,001 a 1.000, igualmente espaçados em escala log;
- a **rede** de uma camada escondida da seção 14.4, com preditores e resposta padronizados, e o número de unidades e o `alpha` escolhidos por validação cruzada na mesma grade de lá: 16 ou 64 unidades e sete valores de `alpha`, de 0,1 a 100;
- a **floresta aleatória** da seção 13.3, com 200 árvores e $m = 4$ colunas sorteadas por divisão, perto de $\sqrt{19}$, a escolha usual, e as colunas como estão, sem padronizar nem buscar nada.

As duas buscas usam os mesmos cinco grupos e o MAE como critério, e cada modelo vira uma função, para que os quatro possam ser ajustados de novo em outras divisões:

In [ ]:
grupos = KFold(5, shuffle=True,
    random_state=14)
metrica = "neg_mean_absolute_error"

def busca(modelo, grade, X, y):
    cv = GridSearchCV(modelo, grade,
        cv=grupos, scoring=metrica)
    return cv.fit(X, y)

grade_lasso = np.logspace(-3, 3, 61)
grade_rede = np.logspace(-1, 2, 7)

def lasso_por_cv(X, y):
    modelo = Pipeline([
        ("escala", StandardScaler()),
        ("lasso", Lasso(max_iter=100_000)),
    ])
    grade = {"lasso__alpha": grade_lasso}
    return busca(modelo, grade, X, y)

caminho = "rede__regressor__"

def rede_por_cv(X, y):
    mlp = MLPRegressor(
        hidden_layer_sizes=(16,),
        max_iter=2000, random_state=14)
    alvo = TransformedTargetRegressor(
        mlp, transformer=StandardScaler())
    modelo = Pipeline([
        ("escala", StandardScaler()),
        ("rede", alvo),
    ])
    grade = {
        caminho + "alpha": grade_rede,
        caminho + "hidden_layer_sizes":
            [(16,), (64,)]}
    return busca(modelo, grade, X, y)

def ajusta_os_quatro(X, y):
    floresta = RandomForestRegressor(
        n_estimators=200, max_features=4,
        random_state=14)
    return {
        "linear": LinearRegression().fit(X, y),
        "lasso": lasso_por_cv(X, y),
        "rede": rede_por_cv(X, y),
        "floresta": floresta.fit(X, y),
    }

> **🔧 Função**
>
> - **`Lasso(max_iter=100_000)`** — o lasso; o `alpha` dele é $\lambda/(2n)$, na forma da seção 11.4, e `max_iter` é o limite de iterações do método de ajuste.
> - **`RandomForestRegressor(n_estimators, max_features, random_state)`** — a floresta aleatória de regressão: `n_estimators` árvores, cada uma numa amostra bootstrap, com `max_features` colunas sorteadas como candidatas em cada divisão; `random_state` fixa os sorteios.

A rede, com centenas de pesos, erra menos que o lasso, que fica com poucos coeficientes? E a floresta, que não pediu padronização nem busca? A tabela traz, para cada modelo, o MAE e o $R^2$ nos 88 jogadores de teste e o número de pesos do modelo, com o intercepto (na rede, os vieses) e, no lasso, só os coeficientes que não são zero:

In [ ]:
modelos = ajusta_os_quatro(
    treino_X, treino_y)
escolha = modelos["rede"].best_params_
camada = escolha[
    caminho + "hidden_layer_sizes"]
alpha_rede = escolha[caminho + "alpha"]
print("rede escolhida:", camada, "e",
    numero_br(alpha_rede))
lasso = modelos["lasso"].best_estimator_[-1]
mlp = modelos["rede"].best_estimator_[-1]
mlp = mlp.regressor_
pesos = {
    "linear": modelos["linear"].coef_.size + 1,
    "lasso": int((lasso.coef_ != 0).sum()) + 1,
    "rede": sum(c.size for c in mlp.coefs_)
        + sum(v.size for v in mlp.intercepts_),
    "floresta": "—",
}
linhas = {}
for nome, modelo in modelos.items():
    prev = modelo.predict(teste_X)
    linhas[nome] = {
        "MAE": mean_absolute_error(teste_y, prev),
        "R²": r2_score(teste_y, prev)}
tabela = pd.DataFrame(linhas).T
mostra = pd.DataFrame({
    "MAE": tabela["MAE"].map(
        lambda v: numero_br(v, 0)),
    "R²": tabela["R²"].map(numero_br),
    "pesos": pd.Series(pesos)})
print(mostra.to_string())
print("menor MAE:", tabela["MAE"].idxmin())
mae_floresta = tabela.loc["floresta", "MAE"]
print("floresta abaixo de 160:",
    mae_floresta < 160)
r2 = tabela["R²"]
dif = abs(r2["rede"] - r2["floresta"])
print("R² da rede e da floresta")
print("  a menos de 0,01:", dif < 0.01)

> **🔧 Função**
>
> - **`pd.DataFrame(linhas).T`** — a tabela com uma coluna por modelo; o `.T` a transpõe, e cada modelo vira uma linha.

Nesta divisão, a busca escolhe a mesma rede da seção 14.4, com 16 unidades e `alpha` 3,16. Ela erra menos que a regressão linear e que o lasso: 178 mil dólares em média, contra 232 e 236. A floresta tem o menor MAE de teste, abaixo de 160. No $R^2$, rede e floresta ficam a menos de 0,01 uma da outra.

O lasso ficou com 8 das 19 colunas, 9 pesos com o intercepto, e a rede tem 337, contra os 20 da regressão linear. A floresta não se resume a uma lista de pesos: são 200 árvores, cada uma com as suas divisões.

### Vinte divisões

São 88 jogadores de teste numa divisão só. Com outro sorteio de treino e teste, a ordem dos quatro muda? Com `random_state` de 0 a 19, tudo se repete em 20 divisões. Em cada uma, o lasso e a rede fazem a sua busca por validação cruzada só com os jogadores de treino dela, e os jogadores de teste não entram em nenhuma escolha. São 20 buscas da rede, e este chunk demora a rodar:

In [ ]:
maes = {}
alphas = {}
alphas_rede = {}
for semente in range(20):
    partes = train_test_split(
        X, y, test_size=1/3,
        random_state=semente)
    tr_X, te_X, tr_y, te_y = partes
    ajustados = ajusta_os_quatro(tr_X, tr_y)
    maes[semente] = {
        nome: mean_absolute_error(
            te_y, m.predict(te_X))
        for nome, m in ajustados.items()}
    lasso_cv = ajustados["lasso"]
    alphas[semente] = lasso_cv.best_params_[
        "lasso__alpha"]
    rede_cv = ajustados["rede"]
    alphas_rede[semente] = (
        rede_cv.best_params_[
            caminho + "alpha"])
maes = pd.DataFrame(maes).T
alphas = pd.Series(alphas)
alphas_rede = pd.Series(alphas_rede)
print("divisões:", len(maes))
em_cima = alphas == grade_lasso.max()
em_baixo = alphas == grade_lasso.min()
print("alpha do lasso na ponta")
print("  de cima da grade:", em_cima.sum())
print("  de baixo da grade:", em_baixo.sum())
dif = maes["lasso"] - maes["linear"]
print("nelas, MAE do lasso e da linear")
print("  a menos de 2:",
    (dif[em_baixo].abs() < 2).all())
print("alpha da rede na ponta")
print("  de cima da grade:",
    (alphas_rede == grade_rede.max()).sum())
print("  de baixo da grade:",
    (alphas_rede == grade_rede.min()).sum())

Na ponta de cima da grade, a das penalidades mais fortes, o `alpha` do lasso não cai em nenhuma divisão. Na de baixo, cai em uma: nela, a validação cruzada prefere a menor penalidade da grade, e o lasso fica a menos de 2 mil dólares de MAE da regressão linear, que é o lasso sem penalidade nenhuma.

A grade da rede também tem pontas. O `alpha` dela não cai na de cima em nenhuma divisão e cai na de baixo em uma: nessa divisão, uma grade que descesse mais poderia escolher uma penalidade ainda menor.

A figura mostra os quatro MAE de teste de cada divisão numa linha. O que olhar é a ordem das cores dentro de cada linha, e se ela se repete de uma linha para a outra:

In [ ]:
# Figura: MAE de teste (em milhares de dólares) dos quatro modelos em cada uma das 20 divisões do `Hitters`, uma divisão por linha, ordenadas de cima para baixo pelo MAE da rede: floresta aleatória (azul), rede (laranja), lasso (verde) e regressão linear (círculo cinza vazado, que deixa ver o verde quando os dois quase coincidem). O traço cinza liga o menor ao maior MAE de cada divisão.
cores = {"floresta": "C0", "rede": "C1",
    "lasso": "C2"}
ordem = maes.sort_values("rede").index
posicao = pd.Series(
    range(len(ordem), 0, -1), index=ordem)
fig, ax = plt.subplots(figsize=(3.8, 4.8))
altura = posicao[maes.index]
ax.hlines(altura, maes.min(axis=1),
    maes.max(axis=1), color="0.45",
    linewidth=0.8, alpha=0.5, zorder=1)
for nome, cor in cores.items():
    ax.scatter(maes[nome], altura,
        s=22, color=cor, label=nome,
        zorder=2, linewidths=0)
ax.scatter(maes["linear"], altura,
    s=40, facecolors="none",
    edgecolors="0.45", linewidths=1.2,
    label="linear", zorder=3)
ax.set_yticks([])
ax.set_ylabel("divisão (pelo MAE da rede)")
ax.set_xlabel("MAE de teste (mil US$)")
ax.xaxis.set_major_formatter(
    lambda v, _: numero_br(v, 0))
ax.legend(loc="lower center",
    bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`maes.sort_values("rede").index`** — os rótulos das divisões, do menor ao maior MAE da rede; **`posicao`** dá a cada divisão a sua altura no gráfico, a do menor MAE em cima.
> - **`maes.min(axis=1)`**, **`maes.max(axis=1)`** — o menor e o maior valor de cada linha, isto é, de cada divisão.
> - **`ax.hlines(y, x0, x1)`** — um traço horizontal na altura `y`, de `x0` a `x1`, um por divisão.

Os números confirmam a figura: o MAE médio das 20 divisões e, para quatro pares de modelos, em quantas divisões o primeiro erra menos:

In [ ]:
print("MAE médio de teste:")
medias = maes.mean()
for nome, media in medias.items():
    print(f"  {nome}: {numero_br(media, 0)}")
print("menor média:", medias.idxmin())
print("floresta abaixo de 170:",
    medias["floresta"] < 170)
pares = [("rede", "linear"),
    ("rede", "lasso"),
    ("floresta", "rede"),
    ("lasso", "linear")]
for a, b in pares:
    n = (maes[a] < maes[b]).sum()
    print(f"{a} < {b}: {n} de {len(maes)}")
folga = maes["rede"] - maes["floresta"]
print("floresta mais de 5 abaixo")
print("  da rede em todas:", (folga > 5).all())
lineares = maes[["lasso", "linear"]]
dist = lineares.min(axis=1) - maes["rede"]
print("lineares mais de 10 acima")
print("  da rede em todas:", (dist > 10).all())
menor = maes.min().min()
maior = maes.max().max()
print("menor MAE:", numero_br(menor, 0))
print("maior MAE:", numero_br(maior, 0))

Nas 20 divisões, a rede erra menos que a regressão linear e que o lasso em todas, por mais de 10 mil dólares em cada uma, e a floresta erra menos que a rede em todas, por mais de 5 mil dólares em cada uma. Entre os lineares, o lasso erra menos que a regressão linear em 17 das 20.

Em média, a floresta erra menos de 170 mil dólares, a rede 196, o lasso 244 e a regressão linear 260. A ordem floresta, rede, modelos lineares se repetiu em todas as divisões; a ordem entre o lasso e a regressão linear, não.

Nestes dados, então, a rede ganha dos modelos lineares e perde para a floresta. O trabalho que a rede pediu compensou contra os lineares, mas não contra a floresta, que, sem padronização e sem busca, errou menos que ela nas 20 divisões.

O princípio da ***navalha de Occam*** manda, entre métodos de desempenho parecido, ficar com o mais simples. Aqui ele não tem o que decidir: a floresta fica abaixo da rede, e a rede abaixo dos lineares, em todas as divisões. Se o lasso e a rede tivessem errado parecido, pesaria também a leitura: na divisão da primeira tabela, os 8 coeficientes do lasso se leem um a um, e os 337 pesos da rede não.

### Onde a rede compensa

O treino dos jogadores tem 175 linhas, com colunas de naturezas diferentes: rebatidas, anos de carreira, liga. As imagens de dígitos da seção 14.2 são outro tipo de dado: todas as colunas são pixels, na mesma escala, e o dígito é uma combinação deles:

In [ ]:
treino = pd.read_csv(
    "dados/mnist_treino.csv.gz")
teste = pd.read_csv(
    "dados/mnist_teste.csv.gz")
dig_X = treino.drop(columns="digito") / 255
dig_y = treino["digito"]
dig_teste_X = teste.drop(
    columns="digito") / 255
dig_teste_y = teste["digito"]
print("imagens de treino:", len(dig_X))
print("colunas:", dig_X.shape[1])
print("imagens de teste:", len(dig_teste_y))

São 6.000 imagens de treino, com 784 colunas, e 2.000 de teste. Nelas, a floresta também ganha da rede, ou a ordem se inverte? E o número de imagens de treino importa? A comparação da seção 14.2 se refaz com as primeiras $n$ imagens de treino, de 150 a 6.000, com a logística e a arquitetura da rede de lá, e agora também com uma floresta de 200 árvores. Uma rede, com uma semente só, pode ter sorte ou azar no sorteio dos pesos iniciais e da ordem em que as imagens passam pelo treino; por isso entram cinco redes, com as sementes de 0 a 4. As configurações não são reescolhidas para cada $n$. O que se conta são os erros nas mesmas imagens de teste:

In [ ]:
tamanhos = (150, 300, 600,
    1500, 3000, 6000)
erros = {}
for n in tamanhos:
    parte_X = dig_X.iloc[:n]
    parte_y = dig_y.iloc[:n]
    dig_modelos = {
        "logística": LogisticRegression(
            C=0.1, max_iter=1000),
        "floresta": RandomForestClassifier(
            n_estimators=200,
            random_state=14)}
    for s in range(5):
        rede = MLPClassifier(
            hidden_layer_sizes=(256, 128),
            random_state=s)
        dig_modelos[f"rede {s}"] = rede
    erros[n] = {}
    for nome, m in dig_modelos.items():
        m.fit(parte_X, parte_y)
        prev = m.predict(dig_teste_X)
        errados = prev != dig_teste_y
        erros[n][nome] = int(errados.sum())
erros = pd.DataFrame(erros).T
redes = erros.filter(like="rede ")
resumo = pd.DataFrame({
    "logística": erros["logística"],
    "floresta": erros["floresta"],
    "rede": redes.mean(axis=1)})
um = lambda v: numero_br(v, 1)
print(resumo.to_string(
    formatters={"rede": um}))

> **🔧 Função**
>
> - **`RandomForestClassifier(n_estimators, random_state)`** — a floresta aleatória de classificação; cada árvore vota numa classe, e a floresta fica com a mais votada. Sem `max_features`, sorteia $\sqrt{p}$ colunas por divisão, a escolha usual para classificação.
> - **`erros.filter(like="rede ")`** — as colunas cujo nome contém `"rede "`, aqui as cinco redes; **`.mean(axis=1)`** tira a média de cada linha, isto é, de cada $n$.
> - **`df.to_string(formatters={"rede": um})`** — a tabela como texto, com a coluna `rede` escrita pela função `um`, com uma casa decimal.

A coluna `rede` é a média dos erros das cinco redes. As perguntas se respondem com comparações linha a linha, e a última delas com a razão entre os erros da rede e os da logística:

In [ ]:
media_redes = resumo["rede"]
outras = resumo[["logística", "floresta"]]
menor_outra = outras.min(axis=1)
print("rede abaixo das outras duas")
print("  em todo n:",
    (media_redes < menor_outra).all())
folga_dig = (resumo["floresta"]
    - media_redes)
print("floresta mais de 5 erros")
print("  acima da rede em todo n:",
    (folga_dig > 5).all())
largura = (redes.max(axis=1)
    - redes.min(axis=1))
print("maior distância entre as redes:",
    int(largura.max()))
quedas = resumo.diff().dropna() < 0
print("as três caem a cada n:",
    quedas.all().all())
razao = media_redes / resumo["logística"]
print("razão rede/logística:")
texto = razao.map(numero_br)
print(texto.to_string())
print("razão > 0,8 (n ≤ 600):",
    (razao.loc[:600] > 0.8).all())
print("razão < 0,8 (n ≥ 1.500):",
    (razao.loc[1500:] < 0.8).all())
vantagem = (resumo["logística"]
    - media_redes)
print("erros a menos que a logística:")
print(vantagem.map(um).to_string())

> **🔧 Função**
>
> - **`serie.loc[:600]`**, **`serie.loc[1500:]`** — as linhas com rótulo até 600 e de 1.500 em diante. Com `.loc`, o rótulo da ponta entra no resultado, ao contrário do `iloc`, em que o fim do intervalo fica de fora.

Na figura, o que olhar é se a linha laranja, da rede, fica abaixo das outras duas em todo $n$, e como a distância entre ela e a cinza, da logística, muda com $n$:

In [ ]:
# Figura: Taxa de erro de teste nas 2.000 imagens de teste do MNIST contra o número $n$ de imagens de treino, em escala log: regressão logística (cinza), floresta aleatória com 200 árvores (azul) e rede com duas camadas escondidas (laranja), esta na média de cinco sementes.
n_teste = len(dig_teste_y)
taxas = resumo / n_teste
fig, ax = plt.subplots(figsize=(3.8, 3.0))
cores = {"logística": "0.45",
    "floresta": "C0", "rede": "C1"}
for nome, cor in cores.items():
    ax.plot(taxas.index, taxas[nome],
        color=cor, marker="o",
        markersize=5, label=nome)
ax.set_xscale("log")
marcas = [150, 600, 1500, 6000]
rotulos = [numero_br(n, 0) for n in marcas]
ax.set_xticks(marcas, rotulos)
ax.minorticks_off()
ax.set_xlabel("imagens de treino (n)")
ax.set_ylabel("taxa de erro de teste")
ax.yaxis.set_major_formatter(
    lambda v, _: numero_br(v, 2))
ax.set_ylim(bottom=0)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`ax.set_xticks(posicoes, rotulos)`** — marca o eixo horizontal só nas posições dadas, com os rótulos dados; **`ax.minorticks_off()`** tira as marcas menores que a escala log acrescenta.

As três curvas caem a cada passo. Com 150 imagens de treino, menos que as 175 linhas de treino dos jogadores, a rede já erra menos que a floresta e que a logística, e isso vale em todo $n$, com a floresta mais de 5 erros acima da média das redes. As cinco redes ficam perto umas das outras: em nenhum $n$ a que mais erra passa a que menos erra por mais de 35 erros.

Nos jogadores, a floresta ganhou da rede nas 20 divisões. O número de linhas, sozinho, não explica essa troca de posição: com menos linhas de treino que nos jogadores, a rede ganhou nas imagens. Os dois problemas, porém, diferem em outras coisas: o tipo das colunas, quantas são (19 e 784), a resposta e a medida de erro, as próprias redes e o $m$ da floresta. Dois conjuntos de dados não dizem qual dessas diferenças pesa.

Se o número de imagens pesa na distância entre a rede e a logística depende de como ela se mede. Em número de imagens, a rede erra 81,2 imagens a menos que a logística com 150 imagens de treino e 94,8 com 6.000, e só 40,2 com 600; na figura, a distância vertical entre a linha laranja e a cinza é parecida nas duas pontas. Em proporção dos erros da logística, a razão entre os erros das duas é 0,88 com 150 imagens e 0,59 com 6.000: fica acima de 0,8 até 600 imagens e abaixo de 0,8 de 1.500 em diante. Nessa medida, a vantagem da rede é pequena com poucas centenas de imagens e maior com milhares.

> **📌 Nota**
>
> **O que as comparações entre modelos desta seção mostram.** Nos jogadores, uma tabela pequena com colunas de naturezas diferentes, a floresta errou menos que a rede em todas as divisões, e a rede, menos que os modelos lineares. Ali, a rede e o lasso foram escolhidos por validação cruzada, e a floresta ficou com a configuração usual, sem busca. Nas imagens, a rede errou menos que a floresta e que a logística em todo $n$, e a razão entre os erros dela e os da logística ficou acima de 0,8 até 600 imagens e abaixo de 0,8 de 1.500 em diante. Ali, nenhum dos três modelos passou por busca: a rede e a logística usam a configuração da seção 14.2, a mesma em todo $n$, e a floresta, a usual.
>
> **A orientação geral, que dois conjuntos de dados não bastam para provar.** Uma rede neural tende a compensar quando a resposta depende de combinações de muitas colunas do mesmo tipo, como os pixels de uma imagem, sobretudo com muitos dados de treino. Em tabelas com colunas de naturezas diferentes, a comparação começa pelos modelos lineares e pelas florestas, que são mais fáceis de ajustar, e os lineares se deixam ler coeficiente a coeficiente.

A rede densa desta seção não usa a vizinhança dos pixels: cada pixel é uma coluna, e nada na rede diz quais colunas são pixels vizinhos. Se as 784 colunas forem embaralhadas numa ordem sorteada, a mesma em todas as imagens de treino e de teste, a rede deve aprender tão bem quanto com os pixels na ordem. A rede de semente 0 é ajustada de novo com os pixels das 6.000 imagens embaralhados, na mesma ordem sorteada em todas:

In [ ]:
rng = np.random.default_rng(14)
ordem = rng.permutation(dig_X.shape[1])
embaralhada = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    random_state=0)
embaralhada.fit(
    dig_X.iloc[:, ordem], dig_y)
prev = embaralhada.predict(
    dig_teste_X.iloc[:, ordem])
erros_emb = int(
    (prev != dig_teste_y).sum())
na_ordem = redes.loc[6000]
print("rede 0, pixels na ordem:",
    na_ordem["rede 0"])
print("rede 0, embaralhados:", erros_emb)
print("5 redes, pixels na ordem:")
print("  de", na_ordem.min(),
    "a", na_ordem.max())
dentro = (na_ordem.min() <= erros_emb
    <= na_ordem.max())
print("dentro dessa faixa:", dentro)

> **🔧 Função**
>
> - **`rng.permutation(k)`** — os inteiros de 0 a `k − 1` numa ordem sorteada pelo gerador `rng`.
> - **`df.iloc[:, ordem]`** — todas as linhas de `df`, com as colunas nas posições dadas por `ordem`, nessa ordem.

Com os pixels embaralhados, a rede de semente 0 erra 132 imagens de teste, contra 141 com os pixels na ordem, e fica dentro da faixa das cinco redes treinadas com os pixels na ordem, de 124 a 146: a diferença não se distingue da que a troca de semente já produz. A rede densa não vê essa estrutura; quando os dados têm vizinhança ou ordem, a candidata é uma rede que a use.

As ***redes convolucionais*** usam a vizinhança dos pixels e são a ferramenta usual para imagens; as ***redes recorrentes*** usam a ordem das palavras de um texto ou dos valores de uma série no tempo. As convolucionais costumam ter muitas camadas; as recorrentes aplicam a mesma camada a cada passo da sequência. As duas se ajustam com bibliotecas próprias de tensores, em geral com placas de vídeo.

## Leituras adicionais

- O capítulo 10 de James et al. (2023), com as redes convolucionais (seção 10.3), as recorrentes (seção 10.5) e a interpolação e a dupla descida (seção 10.8), em que o erro de teste volta a cair quando o número de parâmetros passa do número de observações. O PDF gratuito e os dados estão no [site oficial do livro](https://www.statlearning.com).
- O capítulo 11 de Hastie et al. (2009), "Neural Networks", num tratamento mais técnico do ajuste por retropropagação e da penalidade sobre os pesos.
- Os capítulos 10, "Introduction to Artificial Neural Networks with Keras", e 11, "Training Deep Neural Networks", de Géron (2022), com as mesmas redes numa biblioteca de tensores.
- Goodfellow et al. (2016), o livro de referência sobre aprendizado profundo, com a retropropagação, a regularização, a otimização e as redes convolucionais e recorrentes.
- Rumelhart et al. (1986), o artigo que difundiu a retropropagação e mostrou unidades escondidas aprendendo representações.
- Srivastava et al. (2014), o artigo do *dropout*.
- Kingma e Ba (2015), o artigo do Adam, o método padrão de ajuste do `MLPClassifier` e do `MLPRegressor`.
- LeCun et al. (1998), sobre o reconhecimento de dígitos escritos à mão com redes convolucionais, com a base de onde vem o MNIST.
- [O guia de redes neurais supervisionadas do scikit-learn](https://scikit-learn.org/stable/modules/neural_networks_supervised.html), com o `MLPClassifier`, o `MLPRegressor`, a penalidade `alpha` e as dicas de escala.

## Referências

- **Géron**. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed. O'Reilly Media. 2022.
- **Goodfellow; Bengio; Courville**. *Deep Learning*. MIT Press. 2016.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed. Springer. 2009.
- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.
- **Kingma; Ba**. *Adam: A Method for Stochastic Optimization*. 2015.
- **LeCun; Bottou; Bengio; Haffner**. *Gradient-Based Learning Applied to Document Recognition*. 1998.
- **Rumelhart; Hinton; Williams**. *Learning Representations by Back-Propagating Errors*. 1986.
- **Srivastava; Hinton; Krizhevsky; Sutskever; Salakhutdinov**. *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. 2014.